[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhavierc/NLP_taller_1/blob/main/notebook_taller_4.ipynb)

> **Nota sobre Google Colab:** igual que en taller 1 y 3, este cuaderno está pensado para 2 GPU Tesla T4 (Kaggle). En Colab gratuito (1 GPU) cambia `requested_gpus=1` en el PASO 02 antes de ejecutar.

# Taller 4 — GPT-2 como clasificador de tokens para NER clínico

**Carlos Javier Cepeda, David Salamanca, Jose Milciades Ordoñez**

## Objetivo
Resolvemos el mismo problema de los talleres anteriores — reconocimiento de entidades clínicas (síntomas, enfermedades, procedimientos, fármacos y proteínas) sobre el corpus SPACCC en español — pero reemplazamos los encoders bidireccionales del taller 3 (BETO, RoBERTa) por un decoder autoregresivo preentrenado: **GPT-2 en español** (`mrm8488/spanish-gpt2`), usado como clasificador de tokens mediante `GPT2ForTokenClassification` (un cabezal lineal simple sobre las representaciones de `GPT2Model`).

## Qué cambia y qué no respecto al taller 3
- Modelo: un único checkpoint, `mrm8488/spanish-gpt2`, en vez de los dos checkpoints BERT/RoBERTa del taller 3. No existe actualmente una contraparte de GPT-2 en español preentrenada específicamente en dominio clínico/biomédico, por lo que este taller no incluye una comparación de dominio análoga a la del taller 3 — solo compara estrategias de adaptación (`frozen` vs `fine_tuned`) sobre un mismo checkpoint.
- Arquitectura: GPT-2 es un decoder con atención **causal** (cada posición solo atiende al contexto a su izquierda), a diferencia de BETO/RoBERTa, que son encoders **bidireccionales**. Esta es una diferencia estructural relevante para interpretar el F1 resultante, no solo un cambio de checkpoint.
- Tokenización: mismo esquema que taller 3 — spaCy define las palabras de referencia (sin POS tagging) y el tokenizador de GPT-2 las subdivide con `is_split_into_words=True`. GPT-2 no define `pad_token` ni `cls_token`/`sep_token` por defecto; este cuaderno resuelve ambos casos explícitamente (ver PASO 05).
- Entrenamiento: igual que taller 3, usamos `Trainer` de Hugging Face, comparando `frozen` (solo se entrena el cabezal) y `fine_tuned` (se actualiza todo el modelo). En esta prueba ejecutamos 1 modelo × 2 variantes × 1 semilla —123— = 2 entrenamientos.
- Elementos compartidos: la misma partición fija por documento —semilla 42, 600/150/250 documentos—, la misma política de resolución de solapamientos flat longest, las mismas cinco categorías clínicas y la misma métrica `entity_spans`/`scores_from_counts` (F1 de entidades exactas por documento) que talleres 1, 2 y 3.
- Comparabilidad: las comparaciones con talleres anteriores deben considerar las diferencias de tokenización, arquitectura y evaluación; compartir corpus y partición no basta para garantizar equivalencia completa.


### PASO 01 — Dependencias

Instalamos Transformers, datasets, accelerate. El tokenizador de `es_core_news_sm` define las palabras de referencia que después GPT-2 subdivide en subtokens; se desactivan sus componentes de análisis lingüístico. No se utilizan etiquetas POS. La descarga requiere Internet.


In [1]:
# PASO 01 — Dependencias (Internet activado; ejecutar una vez por sesion)
import os, sys, subprocess, importlib.util

def pip_install(*args):
    """Corre pip capturando su salida: si falla, imprime el error real de pip en
    vez de un CalledProcessError vacio (necesario porque -q oculta el traceback de
    fondo que explica CUAL paquete fallo y por que)."""
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-4000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f'pip install fallo (codigo {result.returncode}): {args}')

# Kaggle a veces trae un pip desactualizado que no sabe generar metadatos para
# paquetes "legacy" (setup.py sin pyproject.toml); actualizarlo primero evita el
# error "python setup.py egg_info did not run successfully".
pip_install('--upgrade', 'pip', 'setuptools', 'wheel')

packages = ['transformers==5.6.0', 'datasets>=5,<6', 'accelerate>=1.1',
            'pandas>=2,<3', 'pyarrow>=14', 'requests>=2.31', 'tqdm>=4.66', 'spacy>=3.8,<4']
if importlib.util.find_spec('torch') is None:
    packages.append('torch>=2.4,<3')
pip_install(*packages)
if importlib.util.find_spec('es_core_news_sm') is None:
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'es_core_news_sm'])
if 'transformers' in sys.modules and sys.modules['transformers'].__version__ != '5.6.0':
    raise RuntimeError('Reinicia la sesión de Kaggle: Transformers se actualizó, pero la versión anterior sigue en memoria.')
print('PASO 01 OK. Dependencias instaladas. Continua con la PASO 02.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 21.7 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
PASO 01 OK. Dependencias instaladas. Continua con la PASO 02.


### PASO 02 — Dos escenarios con semilla 123

GPT-2 ejecuta `frozen` con tasa `1e-3` y `fine_tuned` con tasa `2e-5`. Longitud máxima 512, lote 8 por GPU, hasta 8 épocas y paciencia 2. La comprobación técnica usa 20 pasos sobre un lote pequeño del entrenamiento. La partición conserva semilla 42; el test no participa en selección. Se puede ampliar la lista de semillas después de revisar esta prueba.


In [2]:
# PASO 02 - Prueba inicial comparable de dos variantes de un modelo
import os, json, time, random, math, hashlib, platform, traceback
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import requests, torch
from torch import nn

CFG = dict(
    split_seed=42, training_seeds=[123], fraction=1.0, validation_fraction=0.20,
    max_length=512, overlap_subtokens=64, batch_size=8, patience=2, weight_decay=0.01, warmup_steps=0.1,
    learning_rate={'frozen': 1e-3, 'fine_tuned': 2e-5},
    epochs={'frozen': 8, 'fine_tuned': 8},
    class_weights=False,
    checkpoints={'gpt2': 'mrm8488/spanish-gpt2'},
    variants=['frozen', 'fine_tuned'], smoke_steps=20,
    run_final_test=False, final_test_checkpoint=None, final_test_variant=None, final_test_seed=None,
    requested_gpus=2, allow_cpu_for_debug=False,
)
assert 0 < CFG['fraction'] <= 1
assert 0 < CFG['validation_fraction'] < 1, 'Usa validation_fraction=0.20 para reservar el 20 %.'
assert isinstance(CFG['split_seed'], int) and 0 <= CFG['split_seed'] < 2**32
assert (isinstance(CFG['training_seeds'], list) and len(CFG['training_seeds']) >= 1
        and all(type(s) is int and 0 <= s < 2**32 for s in CFG['training_seeds'])
        and len(set(CFG['training_seeds'])) == len(CFG['training_seeds'])), 'Usa una o más semillas enteras distintas.'
random.seed(CFG['split_seed']); np.random.seed(CFG['split_seed']); torch.manual_seed(CFG['split_seed'])
assert CFG['variants'] and set(CFG['variants']) <= {'frozen', 'fine_tuned'}
assert len(CFG['variants']) == len(set(CFG['variants']))
assert 0 <= CFG['overlap_subtokens'] < CFG['max_length'] - 2
assert CFG['requested_gpus'] in (1, 2), 'requested_gpus debe ser 1 o 2.'
available_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if available_gpus == 0 and CFG['allow_cpu_for_debug']:
    device = torch.device('cpu')
    print('AVISO: modo de depuracion CPU explicito; NO valida CUDA ni las dos T4.')
elif available_gpus < CFG['requested_gpus']:
    raise RuntimeError(f'PASO 02: se requieren {CFG["requested_gpus"]} GPU CUDA y se detectaron '
                       f'{available_gpus}. En Kaggle selecciona GPU T4 x2 y reinicia la sesion. '
                       'Para usar deliberadamente una sola GPU cambia requested_gpus=1.')
else:
    device = torch.device('cuda:0')
    torch.cuda.set_device(device)
    torch.cuda.manual_seed_all(CFG['split_seed'])
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
WORK = ROOT / 'spaccc_transformers'
WORK.mkdir(parents=True, exist_ok=True)
RUN = WORK / (time.strftime('%Y%m%d_%H%M%S') + '_' + str(time.time_ns())[-6:])
RUN.mkdir()
CACHE = WORK / 'cache'; CACHE.mkdir(exist_ok=True)
LABELS = ['CHEMICAL', 'DISEASE', 'PROCEDURE', 'PROTEIN', 'SYMPTOM']
TAGS = ['O'] + [f'{prefix}-{label}' for label in LABELS for prefix in ('B', 'I')]
tag_to_ix = {tag: i for i, tag in enumerate(TAGS)}
id2label = {i: tag for i, tag in enumerate(TAGS)}
label2id = {tag: i for i, tag in enumerate(TAGS)}
IGNORE = -100  # Convencion de Hugging Face: posiciones que no se evaluan (padding, subwords de continuacion, tokens especiales).
START = time.perf_counter()
REPORT = {'config': CFG.copy(), 'run_dir': str(RUN), 'device': str(device),
          'gpu': {'available_count': available_gpus, 'requested': CFG['requested_gpus'],
                  'devices': [{'id': i, 'name': torch.cuda.get_device_name(i),
                               'total_memory_gb': torch.cuda.get_device_properties(i).total_memory / 2**30}
                              for i in range(available_gpus)]},
          'cpu': {'logical_cores': os.cpu_count(), 'machine': platform.machine()},
          'versions': {'python': platform.python_version(), 'torch': str(torch.__version__)}}
(RUN / 'config.json').write_text(json.dumps(REPORT, indent=2), encoding='utf-8')
from html import escape
from IPython.display import Markdown, display

def mostrar_tabla_configuracion(titulo, datos):
    """Muestra los campos anidados como filas de una tabla Markdown."""
    def filas(campos, prefijo=''):
        for clave, valor in campos.items():
            nombre = f'{prefijo}.{clave}' if prefijo else clave
            if isinstance(valor, dict):
                yield from filas(valor, nombre)
            else:
                if isinstance(valor, list):
                    valor = ', '.join(map(str, valor))
                elif valor is None:
                    valor = 'Sin definir'
                elif isinstance(valor, bool):
                    valor = 'Sí' if valor else 'No'
                yield nombre, valor

    def texto(valor):
        return escape(str(valor)).replace('|', '&#124;').replace('\n', '<br>')

    lineas = [f'#### {titulo}', '', '| Parámetro | Valor |', '| --- | --- |']
    lineas.extend(f'| {texto(clave)} | {texto(valor)} |' for clave, valor in filas(datos))
    display(Markdown('\n'.join(lineas)))

display(Markdown('### PASO 02 OK'))
mostrar_tabla_configuracion('Configuración', REPORT['config'])
mostrar_tabla_configuracion('Entorno de ejecución', {
    'run_dir': REPORT['run_dir'], 'device': REPORT['device'],
    'gpu_disponibles': REPORT['gpu']['available_count'],
    'gpu_solicitadas': REPORT['gpu']['requested'],
    'cpu': REPORT['cpu'], 'versiones': REPORT['versions'],
})
for gpu in REPORT['gpu']['devices']:
    mostrar_tabla_configuracion(f"GPU {gpu['id']}", gpu)
# Tablas Markdown sin dependencia adicional de tabulate.
def mostrar_tabla(titulo, columnas, filas):
    def texto(x):
        if x is None:
            return 'No estimable'
        return escape(str(x)).replace('|', '&#124;').replace('\n', '<br>')
    lineas = [f'#### {titulo}', '', '| ' + ' | '.join(map(texto, columnas)) + ' |',
              '| ' + ' | '.join(['---'] * len(columnas)) + ' |']
    lineas.extend('| ' + ' | '.join(map(texto, fila)) + ' |' for fila in filas)
    display(Markdown('\n'.join(lineas)))


### PASO 02 OK

#### Configuración

| Parámetro | Valor |
| --- | --- |
| split_seed | 42 |
| training_seeds | 123 |
| fraction | 1.0 |
| validation_fraction | 0.2 |
| max_length | 512 |
| overlap_subtokens | 64 |
| batch_size | 8 |
| patience | 2 |
| weight_decay | 0.01 |
| warmup_steps | 0.1 |
| learning_rate.frozen | 0.001 |
| learning_rate.fine_tuned | 2e-05 |
| epochs.frozen | 8 |
| epochs.fine_tuned | 8 |
| class_weights | No |
| checkpoints.gpt2 | mrm8488/spanish-gpt2 |
| variants | frozen, fine_tuned |
| smoke_steps | 20 |
| run_final_test | No |
| final_test_checkpoint | Sin definir |
| final_test_variant | Sin definir |
| final_test_seed | Sin definir |
| requested_gpus | 2 |
| allow_cpu_for_debug | No |

#### Entorno de ejecución

| Parámetro | Valor |
| --- | --- |
| run_dir | /kaggle/working/spaccc_transformers/20261002_013232_005910 |
| device | cuda:0 |
| gpu_disponibles | 2 |
| gpu_solicitadas | 2 |
| cpu.logical_cores | 4 |
| cpu.machine | x86_64 |
| versiones.python | 3.12.13 |
| versiones.torch | 2.10.0+cu128 |

#### GPU 0

| Parámetro | Valor |
| --- | --- |
| id | 0 |
| name | Tesla T4 |
| total_memory_gb | 14.56219482421875 |

#### GPU 1

| Parámetro | Valor |
| --- | --- |
| id | 1 |
| name | Tesla T4 |
| total_memory_gb | 14.56219482421875 |

### PASO 03 - Descarga y auditoria de los datos

Reutilizamos exactamente el mismo procedimiento de taller 1: descargamos y dejamos en cache las anotaciones NER, los documentos completos y el conjunto de prueba desde Hugging Face (mismo corpus SPACCC). Comprobamos columnas, categorias, duplicacion de textos entre particiones y correspondencia entre cada offset y el texto original. El analisis exploratorio detallado (distribucion de clases, longitud de entidades, densidad por documento) ya se hizo en taller 1 y no se repite aqui; esta PASO solo audita que la descarga sea consistente antes de tokenizar con los transformers.

In [3]:
# PASO 03 - Descarga, textos completos y auditoria de todas las anotaciones (igual que taller 1)
def download_parquet(repo, split, name):
    """Descarga el parquet `split` del dataset `repo` (Hugging Face) y lo guarda como
    `name` dentro de la carpeta de cache. Si el archivo ya existe en cache lo reutiliza
    sin volver a descargarlo. Reintenta la descarga hasta 3 veces y valida que el
    parquet descargado se pueda leer antes de reemplazar la cache."""
    path = CACHE / name
    if not path.exists():
        url = f'https://huggingface.co/datasets/{repo}/resolve/main/data/{split}-00000-of-00001.parquet'
        print('Descargando:', name, flush=True)
        error = None
        for attempt in range(3):
            try:
                response = requests.get(url, timeout=(15, 120))
                response.raise_for_status()
                tmp = path.with_suffix('.tmp')
                tmp.write_bytes(response.content)
                pd.read_parquet(tmp)  # No guardar una respuesta invalida como cache.
                tmp.replace(path)
                break
            except Exception as exc:
                error = exc
                print(f'Intento {attempt + 1}/3: {type(exc).__name__}: {exc}', flush=True)
        if not path.exists():
            raise RuntimeError('PASO 03: revisa Internet de Kaggle y comparte este error.') from error
    return pd.read_parquet(path)

t0 = time.perf_counter()
df_train = download_parquet('IEETA/SPACCC-Spanish-NER', 'train', 'annotations_train.parquet')
df_test = download_parquet('IEETA/SPACCC-Spanish-NER', 'test', 'annotations_test.parquet')
df_docs = download_parquet('IEETA/SPACCC-documents', 'train', 'documents.parquet')
def document_id(value):
    """Obtiene el identificador de un documento a partir de su nombre de archivo."""
    return Path(str(value)).stem.removeprefix('es-')
required = {'filename', 'label', 'start_span', 'end_span', 'text'}
for frame in (df_train, df_test):
    assert required <= set(frame.columns), 'Columnas de anotacion inesperadas.'
    assert not frame[list(required)].isnull().any().any(), 'Anotaciones con valores nulos.'
    frame['doc_id'] = frame.filename.map(document_id)
    assert set(frame.label) <= set(LABELS), 'Categorias nuevas: revisar TAGS.'
df_docs['doc_id'] = df_docs.filename.map(document_id)
assert not df_docs.doc_id.duplicated().any(), 'Identificadores de documento duplicados.'
assert df_docs.document.map(lambda x: isinstance(x, str) and bool(x)).all()
texts = dict(zip(df_docs.doc_id, df_docs.document))
official_train_ids = sorted(df_train.doc_id.unique())
official_test_ids = sorted(df_test.doc_id.unique())
assert not set(official_train_ids) & set(official_test_ids), 'Fuga: documentos en train y test.'
assert (set(official_train_ids) | set(official_test_ids)) <= texts.keys(), 'Faltan documentos completos.'
bad_offsets, quote_only_differences = [], []
for split, frame in [('train', df_train), ('test', df_test)]:
    for row in frame.itertuples(index=False):
        a, b = int(row.start_span), int(row.end_span)
        actual = texts[row.doc_id][a:b]
        detail = {'split': split, 'doc_id': row.doc_id, 'start': a, 'end': b}
        if not (0 <= a < b <= len(texts[row.doc_id])):
            bad_offsets.append(detail)
        elif actual != row.text:
            if actual.replace(chr(34), '') == row.text.replace(chr(34), ''):
                quote_only_differences.append(detail)
            else:
                bad_offsets.append(detail)
assert not bad_offsets, f'Offsets incompatibles: {len(bad_offsets)}. Ejemplos: {bad_offsets[:5]}'
fingerprint = lambda s: hashlib.sha256(' '.join(s.split()).encode()).hexdigest()
train_hashes = {fingerprint(texts[k]) for k in official_train_ids}
test_hashes = {fingerprint(texts[k]) for k in official_test_ids}
assert not train_hashes & test_hashes, 'Textos duplicados entre train y test: revisar particion.'
REPORT['data'] = {'train_documents': len(official_train_ids), 'test_documents': len(official_test_ids),
                  'train_annotations': len(df_train), 'test_annotations': len(df_test),
                  'offset_errors': len(bad_offsets), 'quote_only_differences': quote_only_differences,
                  'download_audit_seconds': time.perf_counter() - t0,
                  'sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                             for p in CACHE.glob('*.parquet')}}
display(Markdown('### PASO 03 OK'))
mostrar_tabla_configuracion('Resumen de descarga y auditoría', {
    'Documentos de entrenamiento': REPORT['data']['train_documents'],
    'Documentos de prueba': REPORT['data']['test_documents'],
    'Anotaciones de entrenamiento': REPORT['data']['train_annotations'],
    'Anotaciones de prueba': REPORT['data']['test_annotations'],
    'Errores de offsets': REPORT['data']['offset_errors'],
    'Diferencias solo por comillas': len(quote_only_differences),
    'Tiempo de descarga y auditoría (s)': round(REPORT['data']['download_audit_seconds'], 3),
})
mostrar_tabla_configuracion('Huellas SHA-256 de los archivos', REPORT['data']['sha256'])
if quote_only_differences:
    lineas_diferencias = [
        '#### Detalle de diferencias solo por comillas', '',
        '| Partición | Documento | Inicio | Fin |', '| --- | --- | ---: | ---: |',
    ]
    for diferencia in quote_only_differences:
        valores = [escape(str(diferencia[campo])).replace('|', '&#124;').replace('\n', '<br>')
                   for campo in ('split', 'doc_id', 'start', 'end')]
        lineas_diferencias.append('| ' + ' | '.join(valores) + ' |')
    display(Markdown('\n'.join(lineas_diferencias)))

Descargando: annotations_train.parquet
Descargando: annotations_test.parquet
Descargando: documents.parquet


### PASO 03 OK

#### Resumen de descarga y auditoría

| Parámetro | Valor |
| --- | --- |
| Documentos de entrenamiento | 750 |
| Documentos de prueba | 250 |
| Anotaciones de entrenamiento | 33757 |
| Anotaciones de prueba | 11239 |
| Errores de offsets | 0 |
| Diferencias solo por comillas | 2 |
| Tiempo de descarga y auditoría (s) | 2.133 |

#### Huellas SHA-256 de los archivos

| Parámetro | Valor |
| --- | --- |
| annotations_train.parquet | 76c692d7cc49d35d030aeb5c27523862ebd7585bdcf31e596f2f7d6ebe37c033 |
| documents.parquet | d866b787254d0ebdc4e7453ea3fdce25edfb1fc058b536404d1dcf15e5a40f24 |
| annotations_test.parquet | 53bf36209410f657f773cfeb804b08487ab1e8d23e3af9e972cc3927140e1413 |

#### Detalle de diferencias solo por comillas

| Partición | Documento | Inicio | Fin |
| --- | --- | ---: | ---: |
| test | S0004-06142008000700003-1 | 1560 | 1647 |
| test | S1134-80462005000100007-1 | 127 | 145 |

### Inspección de la carga

Revisamos dimensiones y una muestra de anotaciones de entrenamiento. Los conteos por categoría describen menciones anotadas; todavía no incluyen la etiqueta `O`, las etiquetas BIO ni las exclusiones por solapamiento, que se construirán más adelante.

In [4]:
overview = pd.DataFrame([
    {'recurso': 'Anotaciones train', 'filas': len(df_train), 'documentos': len(official_train_ids)},
    {'recurso': 'Anotaciones test', 'filas': len(df_test), 'documentos': len(official_test_ids)},
    {'recurso': 'Textos completos', 'filas': len(df_docs), 'documentos': df_docs.doc_id.nunique()},
])
display(overview)
display(df_train[['doc_id', 'text', 'label', 'start_span', 'end_span']].head())
display(df_train.label.value_counts().rename_axis('categoria').reset_index(name='anotaciones_train'))
# Catálogo completo; la asignación BIO a tokens se realiza en el PASO 05.
lineas_bio = [
    '#### Todas las etiquetas BIO', '',
    '| ID | Etiqueta BIO | Categoría | Significado |',
    '| ---: | --- | --- | --- |',
]
for tag in TAGS:
    if tag == 'O':
        categoria, significado = 'Sin entidad', 'Token fuera de una entidad'
    else:
        prefijo, categoria = tag.split('-', 1)
        significado = 'Inicio de entidad' if prefijo == 'B' else 'Continuación de entidad'
    lineas_bio.append(f'| {tag_to_ix[tag]} | {tag} | {categoria} | {significado} |')
display(Markdown('\n'.join(lineas_bio)))
display(Markdown(
    'Las anotaciones cargadas contienen categorías y offsets. '
    'Las etiquetas BIO se asignan a los tokens en el **PASO 05**. '
    'El valor `IGNORE = -100` excluye posiciones de la evaluación y no es una etiqueta BIO.'
))

overview.to_csv(WORK / 'data_overview.csv', index=False)


,recurso,filas,documentos
0,Anotaciones train,33757,750
1,Anotaciones test,11239,250
2,Textos completos,1000,1000


,doc_id,text,label,start_span,end_span
0,S0004-06142005000500011-1,alergias medicamentosas,DISEASE,50,73
1,S0004-06142005000500011-1,fracturas vertebrales y costales,DISEASE,158,190
2,S0004-06142005000500011-1,intervenido de enfermedad de Dupuytren en mano...,PROCEDURE,192,278
3,S0004-06142005000500011-1,enfermedad de Dupuytren,DISEASE,207,230
4,S0004-06142005000500011-1,Diabetes Mellitus tipo II,DISEASE,280,305


,categoria,anotaciones_train
0,PROCEDURE,11065
1,SYMPTOM,9091
2,DISEASE,8065
3,CHEMICAL,3283
4,PROTEIN,2253


#### Todas las etiquetas BIO

| ID | Etiqueta BIO | Categoría | Significado |
| ---: | --- | --- | --- |
| 0 | O | Sin entidad | Token fuera de una entidad |
| 1 | B-CHEMICAL | CHEMICAL | Inicio de entidad |
| 2 | I-CHEMICAL | CHEMICAL | Continuación de entidad |
| 3 | B-DISEASE | DISEASE | Inicio de entidad |
| 4 | I-DISEASE | DISEASE | Continuación de entidad |
| 5 | B-PROCEDURE | PROCEDURE | Inicio de entidad |
| 6 | I-PROCEDURE | PROCEDURE | Continuación de entidad |
| 7 | B-PROTEIN | PROTEIN | Inicio de entidad |
| 8 | I-PROTEIN | PROTEIN | Continuación de entidad |
| 9 | B-SYMPTOM | SYMPTOM | Inicio de entidad |
| 10 | I-SYMPTOM | SYMPTOM | Continuación de entidad |

Las anotaciones cargadas contienen categorías y offsets. Las etiquetas BIO se asignan a los tokens en el **PASO 05**. El valor `IGNORE = -100` excluye posiciones de la evaluación y no es una etiqueta BIO.

### Interpretación de la inspección de la carga

La inspección de la carga muestra que el corpus contiene **1.000 documentos clínicos**, distribuidos en 750 documentos de entrenamiento con 33.757 anotaciones y 250 documentos de prueba con 11.239 anotaciones. Ambos conjuntos presentan aproximadamente 45 anotaciones por documento. Los 750 documentos corresponden a la partición original de entrenamiento, que posteriormente se divide en 600 documentos de entrenamiento y 150 de validación.

La distribución de las anotaciones de entrenamiento presenta un **desbalance entre categorías**:

| Categoría | Anotaciones | Porcentaje |
|---|---:|---:|
| PROCEDURE | 11.065 | 32,78 % |
| SYMPTOM | 9.091 | 26,93 % |
| DISEASE | 8.065 | 23,89 % |
| CHEMICAL | 3.283 | 9,73 % |
| PROTEIN | 2.253 | 6,67 % |
| **Total** | **33.757** | **100 %** |

Los procedimientos, síntomas y enfermedades concentran aproximadamente el **83,60 %** de las anotaciones. En cambio, las categorías CHEMICAL y PROTEIN tienen menor representación, lo que puede dificultar su aprendizaje. Esta distribución respalda el uso de ponderación de clases y la evaluación por categoría, ya que una métrica global puede ocultar un desempeño bajo en las clases menos frecuentes.

Cada anotación identifica el documento, el fragmento de texto, su categoría y sus posiciones de inicio y fin. Los ejemplos muestran entidades de varias palabras y anotaciones anidadas: «enfermedad de Dupuytren» aparece dentro de un fragmento más amplio etiquetado como procedimiento. Esto evidencia la necesidad de resolver los solapamientos antes de convertir las anotaciones al esquema BIO plano utilizado por el modelo.

El esquema de clasificación incluye **11 etiquetas BIO**: `O`, para tokens fuera de una entidad, y dos etiquetas por cada una de las cinco categorías: `B-`, que indica el inicio, e `I-`, que indica la continuación. La tabla muestra el catálogo completo de etiquetas disponibles; sus frecuencias por token se obtienen después de la tokenización y alineación del paso 5. Por tanto, los conteos presentados corresponden a anotaciones de entidades originales, no a cantidades de tokens BIO.

### PASO 04 — Partición fija por documento

Conservamos el algoritmo, la semilla 42 y la huella SHA-256 del notebook original: 600 documentos de entrenamiento y 150 de validación. Las dos variantes del modelo comparten estos documentos. Los 250 documentos de test no participan en selección de modelos ni hiperparámetros.


In [5]:
# PASO 04 - Particion unica compartida por los dos modelos y sus variantes (sin test)
rng = np.random.default_rng(CFG['split_seed'])
order = rng.permutation(official_train_ids).tolist()
n_sample = min(len(order), max(5, math.ceil(len(order) * CFG['fraction'])))
selected = order[:n_sample]
n_val = max(1, round(n_sample * CFG['validation_fraction']))
val_ids, train_ids = sorted(selected[:n_val]), sorted(selected[n_val:])
assert train_ids and val_ids
assert not set(train_ids) & set(val_ids)
assert not {fingerprint(texts[k]) for k in train_ids} & {fingerprint(texts[k]) for k in val_ids}
split_payload = {'train': train_ids, 'validation': val_ids, 'official_test': official_test_ids}
split_sha256 = hashlib.sha256(json.dumps(split_payload, sort_keys=True).encode()).hexdigest()
REPORT['split'] = {'sample_documents': n_sample, 'train_documents': len(train_ids),
                   'validation_documents': len(val_ids), 'test_used_for_training': False,
                   'split_seed': CFG['split_seed'], 'sha256': split_sha256}
(RUN / 'split.json').write_text(json.dumps({**split_payload, 'split_seed': CFG['split_seed'], 'sha256': split_sha256}, indent=2), encoding='utf-8')
display(Markdown('### PASO 04 — Partición por documento'))
mostrar_tabla_configuracion('Resumen de la partición', {
    'Documentos de la muestra (entrenamiento + validación)': n_sample,
    'Documentos de entrenamiento': len(train_ids),
    'Documentos de validación': len(val_ids),
    'Documentos de prueba reservados': len(official_test_ids),
    'Prueba utilizada para entrenar': REPORT['split']['test_used_for_training'],
    'Semilla de partición': CFG['split_seed'],
})

conteos_muestra = (df_train[df_train.doc_id.isin(selected)].label.value_counts()
                   .reindex(LABELS, fill_value=0))
lineas_muestra = [
    '#### Anotaciones por categoría en la muestra', '',
    '| Categoría | Anotaciones |', '| --- | ---: |',
]
lineas_muestra.extend(f'| {categoria} | {int(cantidad):,} |'
                      for categoria, cantidad in conteos_muestra.items())
lineas_muestra.append(f'| **Total** | **{int(conteos_muestra.sum()):,}** |')
display(Markdown('\n'.join(lineas_muestra)))

TALLER1_SPLIT_SHA256 = 'ab06b4e8dd54e120c2fed93f276f1944a6e67c5227281778c38add2eea4dc497'
mostrar_tabla_configuracion('Verificación de la huella SHA-256', {
    'Partición actual': split_sha256,
    'Partición del taller 1': TALLER1_SPLIT_SHA256,
    'Las huellas coinciden': split_sha256 == TALLER1_SPLIT_SHA256,
})
if CFG['fraction'] == 1.0 and CFG['validation_fraction'] == 0.20:
    assert split_sha256 == TALLER1_SPLIT_SHA256, 'La partición difiere de la referencia.'
if split_sha256 == TALLER1_SPLIT_SHA256:
    display(Markdown('**Verificación correcta:** la partición es idéntica a la del taller 1; '
                     'la comparación entre talleres utiliza los mismos documentos.'))
else:
    display(Markdown('**Aviso:** la huella de partición no coincide con la del taller 1. '
                     'Revisa `split_seed`, `fraction` y `validation_fraction` si esperabas '
                     'la misma partición. La comparación entre talleres dejaría de ser '
                     'sobre el mismo conjunto de documentos.'))

display(Markdown(
    f"**PASO 04 OK.** Los {len(CFG['checkpoints'])} modelos × "
    f"{len(CFG['variants'])} variantes × {len(CFG['training_seeds'])} semillas "
    'usarán estos mismos documentos. El entrenamiento todavía no ha comenzado.'
))


### PASO 04 — Partición por documento

#### Resumen de la partición

| Parámetro | Valor |
| --- | --- |
| Documentos de la muestra (entrenamiento + validación) | 750 |
| Documentos de entrenamiento | 600 |
| Documentos de validación | 150 |
| Documentos de prueba reservados | 250 |
| Prueba utilizada para entrenar | No |
| Semilla de partición | 42 |

#### Anotaciones por categoría en la muestra

| Categoría | Anotaciones |
| --- | ---: |
| CHEMICAL | 3,283 |
| DISEASE | 8,065 |
| PROCEDURE | 11,065 |
| PROTEIN | 2,253 |
| SYMPTOM | 9,091 |
| **Total** | **33,757** |

#### Verificación de la huella SHA-256

| Parámetro | Valor |
| --- | --- |
| Partición actual | ab06b4e8dd54e120c2fed93f276f1944a6e67c5227281778c38add2eea4dc497 |
| Partición del taller 1 | ab06b4e8dd54e120c2fed93f276f1944a6e67c5227281778c38add2eea4dc497 |
| Las huellas coinciden | Sí |

**Verificación correcta:** la partición es idéntica a la del taller 1; la comparación entre talleres utiliza los mismos documentos.

**PASO 04 OK.** Los 1 modelos × 2 variantes × 1 semillas usarán estos mismos documentos. El entrenamiento todavía no ha comenzado.

### PASO 05 — Referencia común y ventanas con contexto

Primero resolvemos las anotaciones sobre palabras de spaCy, con alineación estricta y política de entidad más larga. Este conjunto de referencia se calcula una sola vez y es independiente del tokenizador. Se conservan los offsets de caracteres para evaluar por documento.

El tokenizador de GPT-2 procesa esas palabras con `is_split_into_words=True` y `add_prefix_space=True` (necesario para BPE a nivel de bytes con entradas pretokenizadas, igual que RoBERTa en taller 3). Solo se supervisa el primer subtoken. Si el tokenizador omite una palabra, la preparación falla en vez de reducir silenciosamente el conjunto evaluado.

GPT-2 no define `pad_token` por defecto: se reutiliza `eos_token` como `pad_token` antes de construir el collator. Tampoco define `cls_token`/`sep_token` (ambos son `None`): las ventanas usan `bos_token`/`eos_token` como marca de frontera en su lugar — una marca de límite de ventana, no una señal semántica de inicio/fin de documento.

Las ventanas tienen hasta 512 posiciones, límites de palabras y solapamiento objetivo de 64 subtokens. Cada palabra se asigna a una sola ventana, eligiendo aquella donde dispone de más contexto a ambos lados. El resto de sus apariciones aporta contexto, pero no pérdida ni predicciones duplicadas. La selección no consulta las etiquetas gold. Las entidades que atraviesan ventanas se reconstruyen por documento durante la evaluación.

Se reconstruyen las características para evitar reutilizar la caché anterior. Se guardan auditorías y una huella del gold común. El test solo se prepara si se activa expresamente el paso 11.


In [6]:
# PASO 05 - Gold común por palabras y ventanas con solapamiento real
import spacy
import transformers
from transformers import AutoTokenizer
from tqdm.auto import tqdm

t0 = time.perf_counter()
word_nlp = spacy.load('es_core_news_sm')
word_nlp.disable_pipes(*word_nlp.pipe_names)
REPORT['versions'].update(transformers=transformers.__version__, spacy=spacy.__version__,
                          spacy_model=word_nlp.meta.get('version'))
PREPROCESSING_POLICY = 'shared-spacy-first-subtoken-overlap-owner-document-v1'
tokenizers = {
    key: AutoTokenizer.from_pretrained(checkpoint, use_fast=True, add_prefix_space=True)
    for key, checkpoint in CFG['checkpoints'].items()
}
for key, tok in tokenizers.items():
    if tok.pad_token is None:
        # GPT-2 no define pad_token por defecto; sin esto el padding en lote falla.
        tok.pad_token = tok.eos_token
    has_cls_sep = tok.cls_token_id is not None and tok.sep_token_id is not None
    has_bos_eos = tok.bos_token_id is not None and tok.eos_token_id is not None
    assert tok.is_fast and (has_cls_sep or has_bos_eos), f'{key}: sin tokens de frontera (cls/sep o bos/eos).'
    assert CFG['max_length'] <= tok.model_max_length
    tok.padding_side = 'right'
    mostrar_tabla_configuracion(f'Tokenizador: {key}', {
        'Checkpoint': CFG['checkpoints'][key], 'Vocabulario': tok.vocab_size,
        'Primer subtoken supervisado': True, 'Longitud máxima': CFG['max_length'],
        'Solapamiento objetivo': CFG['overlap_subtokens'],
    })

def prepare_common_documents(ids, annotations):
    """Gold independiente del modelo; offsets estrictos y solapes flat-longest."""
    groups = {k: g for k, g in annotations.groupby('doc_id')}
    documents = {}
    stats = Counter(documents=len(ids), annotations=0, duplicates=0, unaligned=0,
                    overlap_excluded=0, retained=0, words=0, ignored_words=0)
    for doc_id, doc in zip(ids, word_nlp.pipe((texts[k] for k in ids), batch_size=16)):
        # Los espacios aislados no tienen representación léxica en BETO.
        tokens = [t for t in doc if not t.is_space]
        offsets = [(t.idx, t.idx + len(t)) for t in tokens]
        starts = {a: i for i, (a, b) in enumerate(offsets)}
        ends = {b: i + 1 for i, (a, b) in enumerate(offsets)}
        gold, candidates, ignored, seen = [0] * len(tokens), [], [], set()
        for row in groups.get(doc_id, annotations.iloc[:0]).itertuples(index=False):
            a, b, label = int(row.start_span), int(row.end_span), row.label
            stats['annotations'] += 1
            identity = (a, b, label)
            if identity in seen:
                stats['duplicates'] += 1
                continue
            seen.add(identity)
            if a not in starts or b not in ends or ends[b] <= starts[a]:
                stats['unaligned'] += 1
                ignored.append((a, b))
                continue
            candidates.append((a, b, label, starts[a], ends[b]))
        retained = []
        for a, b, label, start, end in sorted(candidates, key=lambda x: (-(x[1]-x[0]), x[0], x[2])):
            if any(gold[i] != 0 for i in range(start, end)):
                stats['overlap_excluded'] += 1
                ignored.append((a, b))
                continue
            gold[start] = tag_to_ix[f'B-{label}']
            gold[start+1:end] = [tag_to_ix[f'I-{label}']] * (end-start-1)
            retained.append([a, b, label])
            stats['retained'] += 1
        for i, (a, b) in enumerate(offsets):
            if gold[i] == 0 and any(a < y and b > x for x, y in ignored):
                gold[i] = IGNORE
        assert tokens, f'Documento vacío: {doc_id}'
        documents[doc_id] = {'doc_id': doc_id, 'words': [t.text for t in tokens],
                             'offsets': offsets, 'labels': gold, 'entities': sorted(retained)}
        stats['words'] += len(tokens)
        stats['ignored_words'] += gold.count(IGNORE)
    assert stats['annotations'] == sum(stats[k] for k in ('duplicates', 'unaligned', 'overlap_excluded', 'retained'))
    return documents, dict(stats)

def common_gold_hash(documents):
    payload = {k: {'offsets': v['offsets'], 'labels': v['labels'], 'entities': v['entities']}
               for k, v in documents.items()}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def build_word_windows(document, tokenizer):
    """Ventanas sin cortar palabras, con una sola posición propietaria por palabra."""
    encoding = tokenizer(document['words'], is_split_into_words=True,
                         add_special_tokens=False, truncation=False, verbose=False)
    ids, word_ids = encoding['input_ids'], encoding.word_ids()
    groups = {}
    for pos, wid in enumerate(word_ids):
        assert wid is not None
        groups.setdefault(wid, []).append(pos)
    assert sorted(groups) == list(range(len(document['words']))), (
        f"El tokenizador omitió palabras de {document['doc_id']}; no se permite cambiar el gold.")
    assert word_ids == sorted(word_ids), 'Orden de palabras inesperado.'
    capacity = CFG['max_length'] - 2
    windows, cursor = [], 0
    while cursor < len(groups):
        end, size = cursor, 0
        while end < len(groups) and size + len(groups[end]) <= capacity:
            size += len(groups[end]); end += 1
        if end == cursor:
            raise ValueError(f"Palabra mayor que la capacidad de ventana: {document['doc_id']}/{cursor}")
        left, right = groups[cursor][0], groups[end-1][-1]+1
        if tokenizer.cls_token_id is not None and tokenizer.sep_token_id is not None:
            boundary_start_id, boundary_end_id = tokenizer.cls_token_id, tokenizer.sep_token_id
        else:
            # GPT-2: cls_token/sep_token son None; bos_token y eos_token apuntan al
            # mismo id <|endoftext|> y se usan aqui solo como marca de frontera de
            # ventana (no como senal semantica de inicio/fin real de documento).
            boundary_start_id, boundary_end_id = tokenizer.bos_token_id, tokenizer.eos_token_id
        windows.append({'doc_id': document['doc_id'],
                        'input_ids': [boundary_start_id] + ids[left:right] + [boundary_end_id],
                        'attention_mask': [1] * (right-left+2),
                        'labels': [IGNORE] * (right-left+2),
                        'prediction_word_ids': [-1] * (right-left+2),
                        'start_word': cursor, 'end_word': end,
                        '_left': left, '_right': right})
        if end == len(groups):
            break
        target = right - CFG['overlap_subtokens']
        # Buscar DENTRO de la ventana actual: el original empezaba en end y no solapaba.
        cursor = next((w for w in range(cursor+1, end) if groups[w][0] >= target), end)
    owners = {}
    for wi, window in enumerate(windows):
        left, right = window['_left'], window['_right']
        for wid in range(window['start_word'], window['end_word']):
            context = min(groups[wid][0] - left, right - 1 - groups[wid][-1])
            if wid not in owners or context > owners[wid][0]:
                owners[wid] = (context, wi, 1 + groups[wid][0] - left)
    for wid, (_, wi, pos) in owners.items():
        windows[wi]['labels'][pos] = document['labels'][wid]
        windows[wi]['prediction_word_ids'][pos] = wid
    for window in windows:
        window.pop('_left'); window.pop('_right')
        assert len(window['input_ids']) <= CFG['max_length']
    # Evitar lotes cuya pérdida no tiene ninguna posición evaluable.
    windows = [w for w in windows if any(y != IGNORE for y in w['labels'])]
    supervised = [wid for w in windows for wid, y in zip(w['prediction_word_ids'], w['labels']) if y != IGNORE]
    expected = [i for i, y in enumerate(document['labels']) if y != IGNORE]
    assert sorted(supervised) == expected, 'Faltan palabras o hay supervisión duplicada.'
    return windows

def make_examples(checkpoint_key, documents, split_name):
    start = time.perf_counter()
    examples = []
    for document in tqdm(documents.values(), desc=f'Ventanas {checkpoint_key} {split_name}'):
        examples.extend(build_word_windows(document, tokenizers[checkpoint_key]))
    stats = {'documents': len(documents), 'sequences': len(examples),
             'supervised_words': sum(y != IGNORE for e in examples for y in e['labels']),
             'retained': sum(len(d['entities']) for d in documents.values()),
             'gold_sha256': common_gold_hash(documents), 'policy': PREPROCESSING_POLICY,
             'seconds_this_run': time.perf_counter() - start}
    (RUN / f'alignment_{checkpoint_key}_{split_name}.json').write_text(json.dumps(stats, indent=2), encoding='utf-8')
    return examples, stats

common_documents, REPORT['common_gold'] = {}, {}
for split_name, ids in [('train', train_ids), ('validation', val_ids)]:
    common_documents[split_name], audit = prepare_common_documents(ids, df_train)
    audit['sha256'] = common_gold_hash(common_documents[split_name])
    REPORT['common_gold'][split_name] = audit
    (RUN / f'common_gold_{split_name}.json').write_text(
        json.dumps(common_documents[split_name], ensure_ascii=False), encoding='utf-8')
    mostrar_tabla_configuracion(f'Referencia común — {split_name}', audit)
train_examples, val_examples, REPORT['alignment'] = {}, {}, {}
for checkpoint_key in CFG['checkpoints']:
    train_examples[checkpoint_key], train_stats = make_examples(checkpoint_key, common_documents['train'], 'train')
    val_examples[checkpoint_key], val_stats = make_examples(checkpoint_key, common_documents['validation'], 'validation')
    REPORT['alignment'][checkpoint_key] = {'train': train_stats, 'validation': val_stats}
    for split_name, stats in [('train', train_stats), ('validation', val_stats)]:
        assert stats['gold_sha256'] == REPORT['common_gold'][split_name]['sha256']
        mostrar_tabla_configuracion(f'{checkpoint_key} — {split_name}', stats)
REPORT['preprocessing_seconds'] = time.perf_counter() - t0
REPORT['preprocessing_policy'] = PREPROCESSING_POLICY
mostrar_tabla('PASO 05 OK — Ventanas por modelo', ['Modelo', 'Entrenamiento', 'Validación'],
              [[k, len(train_examples[k]), len(val_examples[k])] for k in CFG['checkpoints']])


config.json:   0%|          | 0.00/883 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/226 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

...

Ventanas gpt2 train:   0%|          | 0/600 [00:00<?, ?it/s]

Ventanas gpt2 validation:   0%|          | 0/150 [00:00<?, ?it/s]

#### gpt2 — train

| Parámetro | Valor |
| --- | --- |
| documents | 600 |
| sequences | 911 |
| supervised_words | 238187 |
| retained | 22401 |
| gold_sha256 | e93757b22c46fb8079377181928694a20441d5691d7ebe1d046a9e34971e5c34 |
| policy | shared-spacy-first-subtoken-overlap-owner-document-v1 |
| seconds_this_run | 1.8660860499999217 |

#### gpt2 — validation

| Parámetro | Valor |
| --- | --- |
| documents | 150 |
| sequences | 239 |
| supervised_words | 61938 |
| retained | 5861 |
| gold_sha256 | f370133e86e6c1dbd2a537e9295a9506fc992f4a254b387f08a34655860790ab |
| policy | shared-spacy-first-subtoken-overlap-owner-document-v1 |
| seconds_this_run | 0.48264479899989965 |

#### PASO 05 OK — Ventanas por modelo

| Modelo | Entrenamiento | Validación |
| --- | --- | --- |
| gpt2 | 911 | 239 |

### PASO 06 — Datasets y pérdida sin ponderación

Los datasets contienen únicamente entradas del modelo y etiquetas. La correspondencia entre ventanas, palabras y documentos se conserva aparte para las métricas. Usamos la entropía cruzada del modelo, sin pesos por clase; cada palabra evaluable aporta una sola posición supervisada. Las continuaciones, repeticiones de contexto y padding usan `IGNORE=-100`. Para esta comparación inicial se mantiene `class_weights=False`.


In [7]:
# PASO 06 - Datasets sin metadatos y pérdida estándar del modelo
from datasets import Dataset
from transformers import DataCollatorForTokenClassification

assert CFG['class_weights'] is False, 'Esta prueba usa pérdida sin ponderación; mantén class_weights=False.'
MODEL_COLUMNS = ('input_ids', 'attention_mask', 'labels')
def model_features(examples):
    return [{k: e[k] for k in MODEL_COLUMNS} for e in examples]

hf_datasets, collators = {}, {}
REPORT['vocabulary'] = {}
for checkpoint_key in CFG['checkpoints']:
    hf_datasets[checkpoint_key] = {
        'train': Dataset.from_list(model_features(train_examples[checkpoint_key])),
        'validation': Dataset.from_list(model_features(val_examples[checkpoint_key])),
    }
    collators[checkpoint_key] = DataCollatorForTokenClassification(
        tokenizer=tokenizers[checkpoint_key], label_pad_token_id=IGNORE)
    counts = Counter(y for e in train_examples[checkpoint_key] for y in e['labels'] if y != IGNORE)
    common_counts = Counter(y for d in common_documents['train'].values() for y in d['labels'] if y != IGNORE)
    assert counts == common_counts, 'La supervisión por palabra debe ser idéntica entre modelos.'
    REPORT['vocabulary'][checkpoint_key] = {
        'train_tag_counts': {tag: counts[i] for i, tag in enumerate(TAGS)},
        'missing_train_tags': [tag for i, tag in enumerate(TAGS) if not counts[i]],
        'tokenizer_vocab_size': tokenizers[checkpoint_key].vocab_size,
        'train_blocks': len(train_examples[checkpoint_key]),
        'validation_blocks': len(val_examples[checkpoint_key]), 'class_weights': False,
    }
    mostrar_tabla_configuracion(f'{checkpoint_key} — Datasets', {
        'Ventanas de entrenamiento': len(train_examples[checkpoint_key]),
        'Ventanas de validación': len(val_examples[checkpoint_key]),
        'Pérdida': 'Entropía cruzada sin ponderación', 'Padding de etiquetas': IGNORE,
    })
    mostrar_tabla(f'{checkpoint_key} — Etiquetas BIO', ['ID', 'Etiqueta', 'Palabras supervisadas', 'Peso'],
                  [[i, tag, counts[i], 1.0] for i, tag in enumerate(TAGS)])
    if REPORT['vocabulary'][checkpoint_key]['missing_train_tags']:
        mostrar_tabla_configuracion('Etiquetas ausentes', {'Etiquetas': REPORT['vocabulary'][checkpoint_key]['missing_train_tags']})
display(Markdown('### PASO 06 OK\n\nAmbos modelos supervisan exactamente las mismas palabras y etiquetas.'))


#### gpt2 — Datasets

| Parámetro | Valor |
| --- | --- |
| Ventanas de entrenamiento | 911 |
| Ventanas de validación | 239 |
| Pérdida | Entropía cruzada sin ponderación |
| Padding de etiquetas | -100 |

#### gpt2 — Etiquetas BIO

| ID | Etiqueta | Palabras supervisadas | Peso |
| --- | --- | --- | --- |
| 0 | O | 164593 | 1.0 |
| 1 | B-CHEMICAL | 1867 | 1.0 |
| 2 | I-CHEMICAL | 251 | 1.0 |
| 3 | B-DISEASE | 5866 | 1.0 |
| 4 | I-DISEASE | 13097 | 1.0 |
| 5 | B-PROCEDURE | 7253 | 1.0 |
| 6 | I-PROCEDURE | 16851 | 1.0 |
| 7 | B-PROTEIN | 1090 | 1.0 |
| 8 | I-PROTEIN | 336 | 1.0 |
| 9 | B-SYMPTOM | 6325 | 1.0 |
| 10 | I-SYMPTOM | 20658 | 1.0 |

### PASO 06 OK

Ambos modelos supervisan exactamente las mismas palabras y etiquetas.

### PASO 07 — Modelos y evaluación común por documento

Reconstruimos una etiqueta por palabra en cada documento, usando únicamente la ventana propietaria de esa palabra. Después convertimos las entidades BIO a offsets de caracteres y exigimos coincidencia exacta de documento, inicio, fin y categoría. Las continuaciones de subtokens no introducen etiquetas `O` artificiales dentro de entidades. Los huecos del gold realmente excluidos sí separan entidades.

Comprobamos que las entidades reconstruidas del gold coincidan exactamente con las anotaciones retenidas, independientemente del modelo. Se calcula F1 micro y métricas por categoría; la exactitud por palabra es secundaria. Una `I-` sin inicio compatible se trata como inicio de entidad y se registra como transición inválida.


In [8]:
# PASO 07 - Modelos, variante congelada/fine-tuned y metricas de NER
from transformers import (AutoModelForTokenClassification, GPT2ForTokenClassification,
                          Trainer)
from safetensors.torch import load_file as load_safetensors

assert transformers.__version__ == '5.6.0', 'Ejecuta el paso 01 y reinicia la sesión antes de continuar.'

class CanonicalCheckpointMixin:
    def save_pretrained(self, *args, **kwargs):
        # Transformers 5 puede revertir los nombres legacy al guardar; Trainer
        # restaura con load_state_dict y necesita los nombres de la arquitectura.
        kwargs['save_original_format'] = False
        return super().save_pretrained(*args, **kwargs)

class CanonicalGPT2ForTokenClassification(CanonicalCheckpointMixin, GPT2ForTokenClassification):
    pass

MODEL_CLASSES = {'gpt2': CanonicalGPT2ForTokenClassification}

def validate_loading_info(model, info, allow_new_head=False):
    missing = sorted(info.get('missing_keys', []))
    unexpected = sorted(info.get('unexpected_keys', []))
    allowed_missing = {k for k in model.state_dict() if k.startswith('classifier.')} if allow_new_head else set()
    bad_missing = set(missing) - allowed_missing
    # Las cabezas de preentrenamiento no se reutilizan en NER; position_ids era
    # un buffer persistente en checkpoints antiguos, no un parámetro aprendido.
    # lm_head. cubre tambien el head de GPT2LMHeadModel (atado a wte por weight tying).
    # attn.masked_bias: buffer legado de la atencion causal de GPT-2, guardado en
    # checkpoints antiguos (incluido mrm8488/spanish-gpt2); la implementacion
    # actual calcula la mascara causal al vuelo y ya no registra ese buffer.
    allowed_unexpected = lambda k: allow_new_head and (
        k.startswith(('cls.', 'lm_head.', 'bert.pooler.', 'roberta.pooler.'))
        or k in ('bert.embeddings.position_ids', 'roberta.embeddings.position_ids')
        or k.endswith('.attn.masked_bias'))
    bad_unexpected = [k for k in unexpected if not allowed_unexpected(k)]
    if bad_missing or bad_unexpected or info.get('mismatched_keys') or info.get('error_msgs'):
        raise RuntimeError(f'Carga incompleta: faltantes={sorted(bad_missing)}, inesperadas={bad_unexpected}, '
                           f'dimensiones={info.get("mismatched_keys")}, errores={info.get("error_msgs")}')
    for name, parameter in model.named_parameters():
        if not torch.isfinite(parameter).all():
            raise RuntimeError(f'Parámetro no finito al cargar: {name}')
    return {'encoder_completo': True, 'cabezal_nuevo': missing,
            'pesos_de_preentrenamiento_no_usados': unexpected}

def load_model(checkpoint_key):
    model, info = MODEL_CLASSES[checkpoint_key].from_pretrained(
        CFG['checkpoints'][checkpoint_key], num_labels=len(TAGS), id2label=id2label,
        label2id=label2id, dtype=torch.float32, output_loading_info=True)
    audit = validate_loading_info(model, info, allow_new_head=True)
    # mrm8488/spanish-gpt2 publica un tokenizer cuyo vocabulario (incluye
    # <|endoftext|> como token anadido en el id 50265) excede el tamano real de
    # la matriz de embeddings del checkpoint (config.vocab_size=50257); sin este
    # ajuste, cualquier id >= 50257 -incluido el token de frontera de ventana-
    # produce un acceso fuera de rango en la GPU ('device-side assert triggered').
    tokenizer = tokenizers[checkpoint_key]
    original_rows = model.get_input_embeddings().weight.shape[0]
    required_rows = max(tokenizer.get_vocab().values()) + 1
    if required_rows > original_rows:
        model.resize_token_embeddings(required_rows)
        assert torch.isfinite(model.get_input_embeddings().weight).all()
    audit['embedding_rows'] = {'original': original_rows, 'required': required_rows,
                               'resized': required_rows > original_rows}
    REPORT.setdefault('model_loading', {})[checkpoint_key] = audit
    return model

def restore_checkpoint_strict(model, checkpoint_dir):
    """Restauración completa, sin tolerar parámetros faltantes o nombres legacy."""
    directory = Path(checkpoint_dir)
    index_path = directory / 'model.safetensors.index.json'
    if index_path.exists():
        names = sorted(set(json.loads(index_path.read_text())['weight_map'].values()))
    else:
        names = ['model.safetensors']
    state = {}
    for name in names:
        shard = load_safetensors(str(directory / name), device='cpu')
        if state.keys() & shard.keys():
            raise RuntimeError('Claves repetidas entre shards del checkpoint.')
        state.update(shard)
    expected = model.state_dict()
    missing, unexpected = set(expected) - set(state), set(state) - set(expected)
    if missing or unexpected:
        raise RuntimeError(f'Checkpoint incompatible: faltantes={sorted(missing)}, '
                           f'inesperadas={sorted(unexpected)}. No reutilices checkpoints de la corrida anterior.')
    for name, value in state.items():
        if value.shape != expected[name].shape or not torch.isfinite(value).all():
            raise RuntimeError(f'Peso inválido en el checkpoint: {name}')
    model.load_state_dict(state, strict=True)
    return {'restauracion_estricta': True, 'tensores_restaurados': len(state)}

class VerifiedCheckpointTrainer(Trainer):
    def _load_best_model(self):
        # Mismo camino probado en el PASO 08. Este notebook usa un proceso
        # con DataParallel; no configura FSDP, DeepSpeed ni adaptadores PEFT.
        self.checkpoint_restore_audit = restore_checkpoint_strict(self.model, self.state.best_model_checkpoint)

def load_saved_model(checkpoint_dir):
    model, info = AutoModelForTokenClassification.from_pretrained(
        checkpoint_dir, local_files_only=True, dtype=torch.float32, output_loading_info=True)
    validate_loading_info(model, info, allow_new_head=False)
    return model

def apply_variant(model, variant):
    """Aplica la variante `frozen` (congela el encoder base; solo se entrena el
    cabezal de clasificacion) o `fine_tuned` (todos los pesos entrenables)."""
    if variant == 'frozen':
        for param in model.base_model.parameters():
            param.requires_grad = False
    elif variant != 'fine_tuned':
        raise ValueError(f'Variante desconocida: {variant}')
    return model

def entity_spans(tag_ids):
    """Reconstruye spans BIO en una secuencia de palabras.

    Una I- sin entidad compatible inicia una entidad y cuenta como transición
    inválida. Los spans se convierten después a offsets del documento; compartir
    esta función no basta para comparar protocolos con distintos gold o ventanas.
    """
    result, active, begin, invalid = set(), None, None, 0
    for i, idx in enumerate(list(tag_ids) + [0]):
        tag = TAGS[idx] if idx >= 0 else 'O'
        prefix, label = tag.split('-', 1) if tag != 'O' else ('O', None)
        continuing = prefix == 'I' and label == active
        if active is not None and not continuing:
            result.add((begin, i, active)); active = None
        if prefix in ('B', 'I') and not continuing:
            invalid += int(prefix == 'I')
            begin, active = i, label
    return result, invalid

def scores_from_counts(tp, predicted, gold):
    """Calcula precision, recall y F1 a partir de conteos agregados: verdaderos
    positivos (tp), total de entidades predichas y total de entidades reales (gold)
    -- identica a la de taller 1."""
    precision = tp / predicted if predicted else 0.0
    recall = tp / gold if gold else 0.0
    return {'precision': precision, 'recall': recall,
            'f1': 2*precision*recall/(precision+recall) if precision+recall else 0.0,
            'support': gold, 'predicted': predicted}


def reconstruct_word_predictions(predictions, examples, documents):
    """Una predicción por palabra; el orden corresponde al dataset de evaluación."""
    assert len(predictions) == len(examples), 'Orden/tamaño de evaluación incompatible.'
    result = {k: [None] * len(d['words']) for k, d in documents.items()}
    for row, example in zip(predictions, examples):
        assert len(row) >= len(example['input_ids'])
        target = result[example['doc_id']]
        for pos, wid in enumerate(example['prediction_word_ids']):
            if wid < 0:
                continue
            assert target[wid] is None, 'Predicción duplicada para una palabra.'
            target[wid] = int(row[pos])
    for doc_id, doc in documents.items():
        for wid, label in enumerate(doc['labels']):
            if label == IGNORE:
                result[doc_id][wid] = 0
            else:
                assert result[doc_id][wid] is not None, f'Palabra sin predicción: {doc_id}/{wid}'
    return result

def character_entities(tag_ids, offsets):
    spans, invalid = entity_spans(tag_ids)
    return {(offsets[a][0], offsets[b-1][1], label) for a, b, label in spans}, invalid

def make_document_metrics(examples, documents):
    def compute_metrics(eval_pred):
        logits = eval_pred.predictions
        if isinstance(logits, tuple):
            logits = logits[0]
        predictions = np.argmax(logits, axis=-1) if logits.ndim == 3 else logits
        word_predictions = reconstruct_word_predictions(predictions, examples, documents)
        totals = {label: Counter() for label in LABELS}
        invalid = correct = tokens = 0
        for doc_id, doc in documents.items():
            predicted = word_predictions[doc_id]
            gold = [y if y != IGNORE else 0 for y in doc['labels']]
            ps, errors = character_entities(predicted, doc['offsets'])
            gs, gold_errors = character_entities(gold, doc['offsets'])
            assert gold_errors == 0 and gs == {tuple(e) for e in doc['entities']}, 'Gold reconstruido distinto.'
            invalid += errors
            for pred, ref in zip(predicted, doc['labels']):
                if ref != IGNORE:
                    correct += int(pred == ref); tokens += 1
            for label in LABELS:
                totals[label].update(tp=sum(e[2] == label for e in ps & gs),
                                     predicted=sum(e[2] == label for e in ps),
                                     gold=sum(e[2] == label for e in gs))
        micro = scores_from_counts(sum(c['tp'] for c in totals.values()),
                                   sum(c['predicted'] for c in totals.values()),
                                   sum(c['gold'] for c in totals.values()))
        result = {k: v for k, v in micro.items()}
        result.update(accuracy=correct/tokens if tokens else 0.0, invalid_bio_transitions=invalid)
        for label, counts in totals.items():
            metrics = scores_from_counts(counts['tp'], counts['predicted'], counts['gold'])
            for name, value in metrics.items():
                result[f'{name}_{label}'] = value
        return result
    return compute_metrics

def preprocess_logits_for_metrics(logits, labels):
    # Reducir memoria de evaluación: no conservar todos los logits de las ventanas.
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

validation_metrics_fns = {k: make_document_metrics(val_examples[k], common_documents['validation'])
                          for k in CFG['checkpoints']}
REPORT['metric_scope'] = 'F1 micro de entidades exactas por documento y offsets de caracteres, gold común.'
display(Markdown('### PASO 07 OK\n\nModelos y métricas por documento preparados.'))


### PASO 07 OK

Modelos y métricas por documento preparados.

### PASO 08 — Comprobación técnica de los dos escenarios

Se usan exclusivamente ventanas de entrenamiento con entidades. Para cada variante se comprueba:

1. Carga completa del encoder y valores finitos; solo el cabezal NER puede inicializarse desde cero.
2. Pérdida/logits finitos y gradientes no nulos. En `frozen`, ningún peso del encoder cambia; en `fine_tuned`, debe cambiar al menos uno. El cabezal debe actualizarse en ambos casos.
3. Guardado con nombres actuales y restauración estricta mediante el mismo método usado para el mejor checkpoint. Después se carga además una instancia nueva: sus pesos y logits deben coincidir con los anteriores al guardado.
4. Descenso de pérdida y predicciones distintas de `O` después de 20 pasos. Estos dos indicadores se muestran por separado: una alerta no equivale a un error de serialización, pero tampoco permite afirmar que el modelo ya reconoce entidades correctamente.

Los errores de integridad bloquean el paso 09 y se guardan en `smoke_test.json`. Las alertas de aprendizaje quedan visibles y el paso 09 vuelve a comprobar si se predicen entidades en validación. Ninguna prueba de un lote garantiza un buen F1 en documentos nuevos. Los pesos temporales se descartan y el entrenamiento comienza nuevamente desde el checkpoint preentrenado.


In [9]:
# PASO 08 - Carga, gradientes, actualización y guardado/recarga para los dos escenarios
from transformers import set_seed
import gc
import re
import tempfile

def find_normalizable_parameter(model):
    """Un parametro de normalizacion (`.weight`) para perturbar y verificar
    restauracion estricta; generaliza entre BERT/RoBERTa (`LayerNorm.weight`)
    y GPT-2 (`ln_1`/`ln_2`/`ln_f`)."""
    for name, parameter in model.named_parameters():
        if name.endswith('.weight') and ('LayerNorm' in name or re.search(r'\.ln_(\d+|f)\.weight$', name)):
            return parameter
    raise RuntimeError('No se encontro un parametro de normalizacion reconocible.')

def parameter_fingerprint(parameters):
    digest = hashlib.sha256()
    for name, parameter in parameters:
        digest.update(name.encode())
        digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def check_checkpoint_roundtrip(model, batch):
    """Verifica restauración del Trainer y carga independiente, sin entrenamiento."""
    inputs = {k: v for k, v in batch.items() if k != 'labels'}
    model.eval()
    with torch.no_grad():
        before = model(**inputs).logits.detach().cpu()
    reloaded = None
    try:
        with tempfile.TemporaryDirectory(prefix='checkpoint_check_', dir=RUN) as directory:
            model.save_pretrained(directory)
            # Perturbar LayerNorm obliga a comprobar que se restaura realmente;
            # no basta con recargar sobre parámetros que ya eran iguales.
            norm_parameter = find_normalizable_parameter(model)
            with torch.no_grad():
                norm_parameter.add_(0.125)
            audit = restore_checkpoint_strict(model, directory)
            with torch.no_grad():
                restored = model(**inputs).logits.detach().cpu()
            torch.testing.assert_close(restored, before, rtol=1e-5, atol=1e-5)
            reloaded = load_saved_model(directory).to(next(model.parameters()).device)
            reloaded.eval()
            old_state, new_state = model.state_dict(), reloaded.state_dict()
            assert old_state.keys() == new_state.keys()
            assert all(torch.equal(old_state[k], new_state[k]) for k in old_state), 'Los pesos cambiaron al recargar.'
            with torch.no_grad():
                after = reloaded(**inputs).logits.detach().cpu()
            torch.testing.assert_close(after, before, rtol=1e-5, atol=1e-5)
            assert torch.equal(after.argmax(-1), before.argmax(-1)), 'Las etiquetas cambiaron al recargar.'
            return {**audit, 'pesos_identicos': True, 'predicciones_identicas': True,
                    'diferencia_maxima_logits': float((after-before).abs().max())}
    finally:
        del reloaded
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def run_smoke_test(checkpoint_key, variant):
    set_seed(CFG['training_seeds'][0])
    candidates = [e for e in train_examples[checkpoint_key] if any(y > 0 for y in e['labels'])]
    assert candidates, 'No hay entidades supervisadas en entrenamiento.'
    # Evitar un lote dominado por O para el diagnóstico; no altera el dataset real.
    candidates.sort(key=lambda e: sum(y > 0 for y in e['labels']) / sum(y != IGNORE for y in e['labels']), reverse=True)
    batch = collators[checkpoint_key](model_features(candidates[:2]))
    batch = {k: v.to(device) for k, v in batch.items()}
    model = apply_variant(load_model(checkpoint_key), variant).to(device)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                                 lr=CFG['learning_rate'][variant], weight_decay=CFG['weight_decay'])
    model.eval()  # Sin dropout; se permiten gradientes para la prueba del lote.
    encoder_before = parameter_fingerprint(model.base_model.named_parameters())
    head_before = parameter_fingerprint(model.classifier.named_parameters())
    losses = []
    try:
        for _ in range(CFG['smoke_steps']):
            optimizer.zero_grad(set_to_none=True)
            outputs = model(**batch)
            loss = outputs.loss
            assert torch.isfinite(loss) and torch.isfinite(outputs.logits).all(), 'Salida no finita.'
            loss.backward()
            norm = torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad), 1.0, error_if_nonfinite=True)
            assert norm > 0, 'Gradiente nulo.'
            if variant == 'frozen':
                assert all(not p.requires_grad and p.grad is None for p in model.base_model.parameters()), 'Encoder no congelado.'
            else:
                assert any(p.grad is not None and torch.count_nonzero(p.grad) > 0
                           for p in model.base_model.parameters()), 'Encoder sin gradiente en fine_tuned.'
            optimizer.step()
            losses.append(float(loss.item()))
        with torch.no_grad():
            final = model(**batch)
            assert torch.isfinite(final.loss) and torch.isfinite(final.logits).all()
            final_loss = float(final.loss.item())
            predictions = final.logits.argmax(-1)
        valid = batch['labels'] != IGNORE
        predicted_entity_words = int(((predictions > 0) & valid).sum().item())
        entity_words = int(((batch['labels'] > 0) & valid).sum().item())
        entity_correct = int(((predictions == batch['labels']) & (batch['labels'] > 0)).sum().item())
        encoder_changed = encoder_before != parameter_fingerprint(model.base_model.named_parameters())
        head_changed = head_before != parameter_fingerprint(model.classifier.named_parameters())
        assert head_changed, 'El cabezal no se actualizó.'
        assert encoder_changed == (variant == 'fine_tuned'), 'Actualización del encoder incompatible con la variante.'
        roundtrip = check_checkpoint_roundtrip(model, batch)
        alerts = []
        if final_loss >= losses[0]:
            alerts.append('La pérdida final del lote no disminuyó.')
        if not predicted_entity_words:
            alerts.append('El modelo predice solo O en las posiciones supervisadas del lote.')
        elif not entity_correct:
            alerts.append('Predice etiquetas de entidad, pero ninguna coincide con las etiquetas de entidad del lote.')
        return {'passed': True, 'status': 'OK_TECNICO_CON_ALERTAS' if alerts else 'OK_TECNICO',
                'steps': CFG['smoke_steps'], 'learning_rate': CFG['learning_rate'][variant],
                'initial_loss': losses[0], 'final_loss': final_loss, 'min_loss': min(losses+[final_loss]),
                'loss_decreased': final_loss < losses[0], 'encoder_updated': encoder_changed,
                'head_updated': head_changed, 'gold_entity_words': entity_words,
                'predicted_entity_words': predicted_entity_words, 'correct_entity_word_labels': entity_correct,
                'checkpoint': roundtrip, 'alerts': alerts}
    finally:
        del optimizer, model, batch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

REPORT.pop('smoke_test', None)
smoke_results = {}
for checkpoint_key in CFG['checkpoints']:
    for variant in CFG['variants']:
        name = f'{checkpoint_key}_{variant}'
        try:
            result = run_smoke_test(checkpoint_key, variant)
        except Exception as exc:
            smoke_results[name] = {'passed': False, 'status': 'ERROR_TECNICO',
                                   'error': str(exc), 'traceback': traceback.format_exc()}
            REPORT['smoke_test'] = {'passed': False, 'variants': smoke_results}
            (RUN / 'smoke_test.json').write_text(json.dumps(REPORT['smoke_test'], indent=2), encoding='utf-8')
            mostrar_tabla_configuracion(f'ERROR — {name}', smoke_results[name])
            raise
        smoke_results[name] = result
        mostrar_tabla_configuracion(name, result)
        for alert in result['alerts']:
            display(Markdown(f'**Alerta de aprendizaje ({name}):** {alert}'))
REPORT['smoke_test'] = {'passed': all(r['passed'] for r in smoke_results.values()), 'variants': smoke_results}
(RUN / 'smoke_test.json').write_text(json.dumps(REPORT['smoke_test'], indent=2), encoding='utf-8')
display(Markdown('### PASO 08 — Integridad técnica comprobada\n\nRevisa las alertas de aprendizaje anteriores. '
                 'Esta prueba no garantiza un buen F1 de validación. Los pesos temporales se descartan.'))


model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] CanonicalGPT2ForTokenClassification LOAD REPORT from: mrm8488/spanish-gpt2
Key                                     | Status     | 
----------------------------------------+------------+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/150 [00:00<?, ?it/s]

...

[transformers] CanonicalGPT2ForTokenClassification LOAD REPORT from: mrm8488/spanish-gpt2
Key                                     | Status     | 
----------------------------------------+------------+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/150 [00:00<?, ?it/s]

#### gpt2_fine_tuned

| Parámetro | Valor |
| --- | --- |
| passed | Sí |
| status | OK_TECNICO |
| steps | 20 |
| learning_rate | 2e-05 |
| initial_loss | 2.4806783199310303 |
| final_loss | 1.6732243299484253 |
| min_loss | 1.6732243299484253 |
| loss_decreased | Sí |
| encoder_updated | Sí |
| head_updated | Sí |
| gold_entity_words | 104 |
| predicted_entity_words | 143 |
| correct_entity_word_labels | 86 |
| checkpoint.restauracion_estricta | Sí |
| checkpoint.tensores_restaurados | 150 |
| checkpoint.pesos_identicos | Sí |
| checkpoint.predicciones_identicas | Sí |
| checkpoint.diferencia_maxima_logits | 0.0 |
| alerts |  |

### PASO 08 — Integridad técnica comprobada

Revisa las alertas de aprendizaje anteriores. Esta prueba no garantiza un buen F1 de validación. Los pesos temporales se descartan.

### PASO 09 — Dos entrenamientos comparables

GPT-2 se ejecuta con `frozen` (`1e-3`) y `fine_tuned` (`2e-5`), semilla 123. Ambas variantes usan las mismas palabras y entidades, hasta 512 posiciones, lote 8 por GPU, 8 épocas máximas, paciencia 2 y warmup del 10 %.

Se guardan nombres actuales de parámetros y el mejor checkpoint se restaura de forma estricta. Se registra la auditoría de recuperación. Se avisa explícitamente si una evaluación produce cero entidades. Pérdida, precisión, recall y F1 quedan en Markdown y en los historiales CSV. Los dos entrenamientos se realizan en Kaggle; no se usa test.

**Nota:** `fp16=torch.cuda.is_available()` combinado con `learning_rate['frozen']=1e-3` sobre un encoder causal GPT-2 es una combinación no probada previamente en este repo. Si aparecen pérdidas no finitas, `CheckFiniteTraining` detiene la corrida con un mensaje claro; en ese caso, considera reducir la tasa de `frozen`.


In [10]:
# PASO 09 - Dos escenarios: un modelo por dos estrategias
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, TrainerCallback, set_seed

assert REPORT.get('smoke_test', {}).get('passed'), 'Ejecuta primero el paso 08.'
for key in CFG['checkpoints']:
    for variant in CFG['variants']:
        probe = REPORT['smoke_test']['variants'].get(f'{key}_{variant}', {})
        assert probe.get('passed') and probe.get('learning_rate') == CFG['learning_rate'][variant], (
            'La prueba técnica no corresponde a esta configuración; repite el paso 08.')

class CheckFiniteTraining(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and metrics.get('eval_predicted') == 0:
            display(Markdown(f"**Alerta en época {state.epoch}:** cero entidades predichas en validación. "
                             'El F1 es cero; revisa esta corrida antes de interpretar la comparación.'))

    def on_log(self, args, state, control, logs=None, **kwargs):
        # FP16 puede omitir pasos por desbordamiento de gradientes; se registra grad_norm.
        for name in ('loss', 'eval_loss'):
            if logs and name in logs and not math.isfinite(float(logs[name])):
                raise RuntimeError(f'{name} no finito. La corrida se detiene; revisa learning_rate.')

def extract_metrics(metrics):
    return {'micro': {k: float(metrics[f'eval_{k}']) for k in ('precision', 'recall', 'f1')}
            | {'support': int(metrics['eval_support']), 'predicted': int(metrics['eval_predicted'])},
            'per_label': {label: {k: float(metrics[f'eval_{k}_{label}'])
                                 for k in ('precision', 'recall', 'f1')}
                          | {'support': int(metrics[f'eval_support_{label}'])} for label in LABELS},
            'loss': float(metrics['eval_loss']), 'accuracy': float(metrics['eval_accuracy']),
            'invalid_bio_transitions': int(metrics['eval_invalid_bio_transitions'])}

def train_run(checkpoint_key, variant, training_seed):
    run_dir = RUN / checkpoint_key / variant / f'seed_{training_seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    set_seed(training_seed)
    model = apply_variant(load_model(checkpoint_key), variant)
    parameters = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    args = TrainingArguments(
        output_dir=str(run_dir), seed=training_seed, data_seed=training_seed,
        num_train_epochs=CFG['epochs'][variant], learning_rate=CFG['learning_rate'][variant],
        per_device_train_batch_size=CFG['batch_size'], per_device_eval_batch_size=CFG['batch_size'],
        weight_decay=CFG['weight_decay'], warmup_steps=CFG['warmup_steps'],
        fp16=torch.cuda.is_available(), max_grad_norm=1.0,
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model='f1', greater_is_better=True,
        logging_strategy='steps', logging_steps=10, logging_first_step=True,
        logging_nan_inf_filter=False, report_to=[], disable_tqdm=False,
    )
    trainer = VerifiedCheckpointTrainer(model=model, args=args,
                      train_dataset=hf_datasets[checkpoint_key]['train'],
                      eval_dataset=hf_datasets[checkpoint_key]['validation'],
                      data_collator=collators[checkpoint_key], processing_class=tokenizers[checkpoint_key],
                      compute_metrics=validation_metrics_fns[checkpoint_key],
                      preprocess_logits_for_metrics=preprocess_logits_for_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG['patience']), CheckFiniteTraining()])
    mostrar_tabla_configuracion(f'Entrenando {checkpoint_key} / {variant}', {
        'Semilla': training_seed, 'Tasa inicial': CFG['learning_rate'][variant],
        'Parámetros entrenables': trainable, 'Parámetros totales': parameters,
        'Épocas máximas': CFG['epochs'][variant], 'Lote por GPU': CFG['batch_size'],
        'Ventanas train': len(train_examples[checkpoint_key]), 'Ventanas validación': len(val_examples[checkpoint_key]),
    })
    try:
        train_output = trainer.train()
        metrics = trainer.evaluate()
        best_dir = run_dir / 'best_model'
        trainer.save_model(str(best_dir))
        best_step = int(Path(trainer.state.best_model_checkpoint).name.rsplit('-', 1)[-1])
        best_epoch = next((r.get('epoch') for r in trainer.state.log_history
                           if r.get('step') == best_step and 'eval_f1' in r), None)
        history = [{k: (str(v) if isinstance(v, float) and not math.isfinite(v) else v)
                    for k, v in row.items() if isinstance(v, (int, float, str, bool))}
                   for row in trainer.state.log_history]
        result = {'checkpoint': checkpoint_key, 'variant': variant, 'seed': training_seed,
                  'learning_rate': CFG['learning_rate'][variant], 'split_sha256': split_sha256,
                  'gold_sha256': REPORT['common_gold']['validation']['sha256'],
                  'parameters': parameters, 'trainable_parameters': trainable,
                  'checkpoint_dir': str(best_dir), 'best_epoch': best_epoch,
                  'best_metric_f1': float(trainer.state.best_metric),
                  'validation': extract_metrics(metrics),
                  'train_runtime_seconds': float(train_output.metrics['train_runtime']),
                  'epochs_completed': float(trainer.state.epoch), 'optimizer_steps': trainer.state.global_step,
                  'history': history, 'checkpoint_restore': trainer.checkpoint_restore_audit}
        (run_dir / 'result.json').write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
        pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
        eval_rows = [r for r in history if 'eval_f1' in r]
        mostrar_tabla(f'{checkpoint_key} — Evolución de validación',
                      ['Época', 'Pérdida', 'Precisión %', 'Recall %', 'F1 %'],
                      [[round(r['epoch'], 2), round(r['eval_loss'], 4), round(100*r['eval_precision'], 2),
                        round(100*r['eval_recall'], 2), round(100*r['eval_f1'], 2)] for r in eval_rows])
        return result
    finally:
        del trainer, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

REPORT['experiments'] = {}
for checkpoint_key in CFG['checkpoints']:
    REPORT['experiments'][checkpoint_key] = {}
    for variant in CFG['variants']:
        REPORT['experiments'][checkpoint_key][variant] = {}
        for training_seed in CFG['training_seeds']:
            result = train_run(checkpoint_key, variant, training_seed)
            REPORT['experiments'][checkpoint_key][variant][str(training_seed)] = result
            (RUN / 'experiments_progress.json').write_text(json.dumps(REPORT['experiments'], indent=2), encoding='utf-8')
n_runs = len(CFG['checkpoints']) * len(CFG['variants']) * len(CFG['training_seeds'])
display(Markdown(f'### PASO 09 OK\n\nFinalizaron **{n_runs} entrenamientos**. Continúa con el paso 10.'))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] CanonicalGPT2ForTokenClassification LOAD REPORT from: mrm8488/spanish-gpt2
Key                                     | Status     | 
----------------------------------------+------------+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### Entrenando gpt2 / frozen

| Parámetro | Valor |
| --- | --- |
| Semilla | 123 |
| Tasa inicial | 0.001 |
| Parámetros entrenables | 8459 |
| Parámetros totales | 124455179 |
| Épocas máximas | 8 |
| Lote por GPU | 8 |
| Ventanas train | 911 |
| Ventanas validación | 239 |

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 50265, 'bos_token_id': 50265, 'pad_token_id': 50265}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Support,Predicted,Accuracy,Invalid Bio Transitions,Precision Chemical,Recall Chemical,F1 Chemical,Support Chemical,Predicted Chemical,Precision Disease,Recall Disease,F1 Disease,Support Disease,Predicted Disease,Precision Procedure,Recall Procedure,F1 Procedure,Support Procedure,Predicted Procedure,Precision Protein,Recall Protein,F1 Protein,Support Protein,Predicted Protein,Precision Symptom,Recall Symptom,F1 Symptom,Support Symptom,Predicted Symptom
1,2.035742,1.713389,0.116393,0.093158,0.103487,5861,4691,0.720753,3285,0.571429,0.041594,0.077544,577,42,0.095767,0.167505,0.121862,1391,2433,0.150273,0.144281,0.147216,1906,1830,0.000000,0.000000,0.000000,305,0,0.036269,0.008323,0.013540,1682,386
2,1.405065,1.370781,0.203787,0.255247,0.226632,5861,7341,0.764749,3521,0.594360,0.474870,0.527938,577,461,0.174076,0.301222,0.220642,1391,2407,0.227515,0.290661,0.255241,1906,2435,0.381679,0.163934,0.229358,305,131,0.104352,0.118312,0.110894,1682,1907
3,1.347563,1.298807,0.210642,0.293807,0.245369,5861,8175,0.775889,3944,0.590998,0.523397,0.555147,577,511,0.191993,0.320633,0.240172,1391,2323,0.233738,0.333683,0.274908,1906,2721,0.316038,0.219672,0.259188,305,212,0.112542,0.161118,0.132518,1682,2408
4,1.305458,1.264646,0.221240,0.303191,0.255812,5861,8032,0.779231,3508,0.627329,0.525130,0.571698,577,483,0.204787,0.332135,0.253359,1391,2256,0.238008,0.341028,0.280354,1906,2731,0.350000,0.275410,0.308257,305,240,0.119724,0.165279,0.138861,1682,2322
5,1.298011,1.245310,0.222356,0.314963,0.260679,5861,8302,0.782557,3745,0.626243,0.545927,0.583333,577,503,0.210240,0.333573,0.257921,1391,2207,0.237491,0.353620,0.284148,1906,2838,0.360324,0.291803,0.322464,305,247,0.121260,0.180737,0.145142,1682,2507
6,1.236339,1.233555,0.226813,0.317523,0.264610,5861,8205,0.783768,3638,0.623077,0.561525,0.590702,577,520,0.209071,0.347951,0.261198,1391,2315,0.253840,0.346800,0.293126,1906,2604,0.382488,0.272131,0.318008,305,217,0.121224,0.183710,0.146065,1682,2549
7,1.290531,1.228085,0.225983,0.330319,0.268367,5861,8567,0.786076,3866,0.623574,0.568458,0.594742,577,526,0.210570,0.366643,0.267506,1391,2422,0.250643,0.357817,0.294791,1906,2721,0.400844,0.311475,0.350554,305,237,0.120631,0.190844,0.147824,1682,2661
8,1.285696,1.225486,0.228311,0.324177,0.267926,5861,8322,0.785172,3709,0.630058,0.566724,0.596715,577,519,0.213328,0.354421,0.266343,1391,2311,0.250094,0.350472,0.291894,1906,2671,0.400000,0.301639,0.343925,305,230,0.123504,0.190250,0.149778,1682,2591


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Support,Predicted,Accuracy,Invalid Bio Transitions,Precision Chemical,Recall Chemical,F1 Chemical,Support Chemical,Predicted Chemical,Precision Disease,Recall Disease,F1 Disease,Support Disease,Predicted Disease,Precision Procedure,Recall Procedure,F1 Procedure,Support Procedure,Predicted Procedure,Precision Protein,Recall Protein,F1 Protein,Support Protein,Predicted Protein,Precision Symptom,Recall Symptom,F1 Symptom,Support Symptom,Predicted Symptom
1.060691,1.090354,8,0.269482,0.373486,0.313072,5861,8123,0.815218,3372,0.582222,0.454073,0.510224,577,450,0.279155,0.437096,0.340712,1391,2178,0.330896,0.451207,0.381798,1906,2599,0.450704,0.104918,0.170213,305,71,0.151150,0.253864,0.189483,1682,2825


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

#### gpt2 — Evolución de validación

| Época | Pérdida | Precisión % | Recall % | F1 % |
| --- | --- | --- | --- | --- |
| 1.0 | 2.7766 | 4.51 | 0.19 | 0.36 |
| 2.0 | 1.6642 | 7.59 | 7.61 | 7.6 |
| 3.0 | 1.3593 | 16.95 | 20.92 | 18.72 |
| 4.0 | 1.2195 | 21.55 | 29.09 | 24.76 |
| 5.0 | 1.1512 | 24.42 | 34.04 | 28.44 |
| 6.0 | 1.1131 | 26.17 | 35.08 | 29.98 |
| 7.0 | 1.0954 | 26.72 | 36.82 | 30.97 |
| 8.0 | 1.0904 | 26.95 | 37.35 | 31.31 |
| 8.0 | 1.0904 | 26.95 | 37.35 | 31.31 |

### PASO 09 OK

Finalizaron **2 entrenamientos**. Continúa con el paso 10.

### PASO 10 — Comparación frozen y fine_tuned

Mostramos precisión, recall y F1 por documento, tiempo y mejor época para cada corrida. Se verifica la misma huella del gold y el mismo soporte por categoría entre variantes. Con una semilla se informa el resultado individual de cada variante y la diferencia fine_tuned − frozen; no se inventa una desviación estándar. Si se amplían las semillas, se calcula media y desviación muestral. A diferencia del taller 3, no hay un segundo checkpoint con el cual comparar: este taller usa un único modelo GPT-2.


In [11]:
# PASO 10 - Comparación válida para una o varias semillas y variantes
experiments = REPORT.get('experiments', {})
assert set(experiments) == set(CFG['checkpoints']), 'Faltan modelos: ejecuta el paso 09.'
rows, label_rows = [], []
for checkpoint_key in CFG['checkpoints']:
    assert set(experiments[checkpoint_key]) == set(CFG['variants'])
    for variant in CFG['variants']:
        assert set(experiments[checkpoint_key][variant]) == {str(s) for s in CFG['training_seeds']}
        for seed in CFG['training_seeds']:
            result = experiments[checkpoint_key][variant][str(seed)]
            assert result['seed'] == seed and result['split_sha256'] == split_sha256
            assert result['gold_sha256'] == REPORT['common_gold']['validation']['sha256']
            metrics = result['validation']['micro']
            assert metrics['support'] == REPORT['common_gold']['validation']['retained']
            rows.append({'checkpoint': checkpoint_key, 'variante': variant, 'semilla': seed,
                         'learning_rate': result['learning_rate'], 'mejor_epoca': result['best_epoch'],
                         'epocas_ejecutadas': result['epochs_completed'], 'pasos': result['optimizer_steps'],
                         'precision_pct': 100*metrics['precision'], 'recall_pct': 100*metrics['recall'],
                         'f1_pct': 100*metrics['f1'], 'train_segundos': result['train_runtime_seconds']})
            for label in LABELS:
                m = result['validation']['per_label'][label]
                label_rows.append({'checkpoint': checkpoint_key, 'variante': variant, 'semilla': seed,
                                   'categoria': label, 'entidades_validacion': m['support'],
                                   'precision_pct': 100*m['precision'], 'recall_pct': 100*m['recall'],
                                   'f1_pct': 100*m['f1']})
comparison, per_label = pd.DataFrame(rows), pd.DataFrame(label_rows)
assert (per_label.groupby('categoria').entidades_validacion.nunique() == 1).all()
aggregate_rows = []
for (key, variant), group in comparison.groupby(['checkpoint', 'variante'], sort=False):
    row = {'checkpoint': key, 'variante': variant, 'n_semillas': len(group)}
    for metric in ['precision_pct', 'recall_pct', 'f1_pct', 'train_segundos']:
        row[metric + '_media'] = float(group[metric].mean())
        row[metric + '_std'] = float(group[metric].std(ddof=1)) if len(group) > 1 else None
    aggregate_rows.append(row)
aggregate = pd.DataFrame(aggregate_rows)
paired_variant_rows = []
if {'frozen', 'fine_tuned'} <= set(CFG['variants']):
    for key in CFG['checkpoints']:
        table = comparison[comparison.checkpoint == key].pivot(index='semilla', columns='variante', values='f1_pct')
        for seed in CFG['training_seeds']:
            paired_variant_rows.append({'checkpoint': key, 'semilla': seed,
                                        'diferencia_pp': float(table.loc[seed, 'fine_tuned'] - table.loc[seed, 'frozen'])})
paired_variant = pd.DataFrame(paired_variant_rows, columns=['checkpoint', 'semilla', 'diferencia_pp'])
for name, table in [('comparison_runs', comparison), ('comparison_aggregate', aggregate),
                    ('comparison_per_label_runs', per_label),
                    ('comparison_paired_variant', paired_variant)]:
    table.to_csv(RUN / f'{name}.csv', index=False)
    if not table.empty:
        shown = table.round(3).astype(object).where(pd.notna(table), None)
        mostrar_tabla(name, list(shown.columns), shown.itertuples(index=False, name=None))
REPORT['comparison'] = {'per_run': rows, 'aggregate': aggregate_rows, 'per_label': label_rows,
                        'paired_variant': paired_variant_rows,
                        'note': 'Una semilla: prueba exploratoria; varias: media y desviación muestral, sin prueba de significancia.'}
REPORT['elapsed_seconds_since_cell02'] = time.perf_counter() - START
REPORT['status'] = 'COMPARACION_CONFIGURADA_COMPLETADA'
(RUN / 'summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
if len(CFG['training_seeds']) == 1:
    display(Markdown('**Prueba de una semilla:** la desviación estándar no es estimable. '
                     'La diferencia observada entre modelos requiere confirmación con más semillas.'))
display(Markdown(f'### PASO 10 OK\n\nResultados y curvas guardados en `{RUN}`. Test reservado.'))


#### comparison_runs

| checkpoint | variante | semilla | learning_rate | mejor_epoca | epocas_ejecutadas | pasos | precision_pct | recall_pct | f1_pct | train_segundos |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| gpt2 | frozen | 123 | 0.001 | 7.0 | 8.0 | 456 | 22.598 | 33.032 | 26.837 | 210.576 |
| gpt2 | fine_tuned | 123 | 0.0 | 8.0 | 8.0 | 456 | 26.948 | 37.349 | 31.307 | 494.009 |

#### comparison_aggregate

| checkpoint | variante | n_semillas | precision_pct_media | precision_pct_std | recall_pct_media | recall_pct_std | f1_pct_media | f1_pct_std | train_segundos_media | train_segundos_std |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| gpt2 | frozen | 1 | 22.598 | No estimable | 33.032 | No estimable | 26.837 | No estimable | 210.576 | No estimable |
| gpt2 | fine_tuned | 1 | 26.948 | No estimable | 37.349 | No estimable | 31.307 | No estimable | 494.009 | No estimable |

#### comparison_per_label_runs

| checkpoint | variante | semilla | categoria | entidades_validacion | precision_pct | recall_pct | f1_pct |
| --- | --- | --- | --- | --- | --- | --- | --- |
| gpt2 | frozen | 123 | CHEMICAL | 577 | 62.357 | 56.846 | 59.474 |
| gpt2 | frozen | 123 | DISEASE | 1391 | 21.057 | 36.664 | 26.751 |
| gpt2 | frozen | 123 | PROCEDURE | 1906 | 25.064 | 35.782 | 29.479 |
| gpt2 | frozen | 123 | PROTEIN | 305 | 40.084 | 31.148 | 35.055 |
| gpt2 | frozen | 123 | SYMPTOM | 1682 | 12.063 | 19.084 | 14.782 |
| gpt2 | fine_tuned | 123 | CHEMICAL | 577 | 58.222 | 45.407 | 51.022 |
| gpt2 | fine_tuned | 123 | DISEASE | 1391 | 27.916 | 43.71 | 34.071 |
| gpt2 | fine_tuned | 123 | PROCEDURE | 1906 | 33.09 | 45.121 | 38.18 |
| gpt2 | fine_tuned | 123 | PROTEIN | 305 | 45.07 | 10.492 | 17.021 |
| gpt2 | fine_tuned | 123 | SYMPTOM | 1682 | 15.115 | 25.386 | 18.948 |

#### comparison_paired_variant

| checkpoint | semilla | diferencia_pp |
| --- | --- | --- |
| gpt2 | 123 | 4.471 |

**Prueba de una semilla:** la desviación estándar no es estimable. La diferencia observada entre modelos requiere confirmación con más semillas.

### PASO 10 OK

Resultados y curvas guardados en `/kaggle/working/spaccc_transformers/20261002_013232_005910`. Test reservado.

In [12]:

from pathlib import Path

encontrados = []
for carpeta in ['/kaggle/working', '/kaggle/input']:
  raiz = Path(carpeta)
  if raiz.exists():
      for nombre in ['result.json', 'experiments_progress.json', 'summary.json']:
          encontrados.extend(raiz.rglob(nombre))

if encontrados:
  for archivo in sorted(encontrados):
      print(archivo)
else:
  print('No se encontraron resultados en esta sesión.')


/kaggle/working/spaccc_transformers/20261002_013232_005910/experiments_progress.json
/kaggle/working/spaccc_transformers/20261002_013232_005910/gpt2/fine_tuned/seed_123/result.json
/kaggle/working/spaccc_transformers/20261002_013232_005910/gpt2/frozen/seed_123/result.json
/kaggle/working/spaccc_transformers/20261002_013232_005910/summary.json


### PASO 11 — Test opcional, reservado

Desactivado por defecto. Actívalo únicamente después de fijar modelo, variante y semilla, usando las claves `final_test_*` del paso 02. Emplea la misma preparación por palabras y la misma evaluación por documento. No se selecciona automáticamente una combinación ni se usan resultados de test para ajustar la tasa de aprendizaje.


In [13]:
# PASO 11 - Evaluación final de GPT-2 fine_tuned; no realiza entrenamiento
if 'CFG' not in globals() or 'REPORT' not in globals():
    raise RuntimeError('La sesión no conserva CFG/REPORT. Recupera los resultados y el entorno antes de evaluar; '
                       'no basta con ejecutar el paso 02 de nuevo.')
# Seleccion explicita del usuario tras comparar el F1 de validacion del PASO 10.
# NOTA: 'fine_tuned' se deja como valor por defecto porque en talleres 1-3 la
# variante con ajuste completo siempre supero a la variante congelada; CONFIRMAR
# contra la tabla real del PASO 10 de esta corrida antes de ejecutar este paso.
CFG.update(run_final_test=True, final_test_checkpoint='gpt2',
           final_test_variant='fine_tuned', final_test_seed=123)
REPORT['config'].update({k: CFG[k] for k in (
    'run_final_test', 'final_test_checkpoint', 'final_test_variant', 'final_test_seed')})
if not CFG['run_final_test']:
    display(Markdown('### PASO 11 OMITIDO\n\nTest reservado; la comparación usa validación.'))
else:
    assert CFG['fraction'] == 1.0
    checkpoint_key, variant, seed = (CFG['final_test_checkpoint'], CFG['final_test_variant'], CFG['final_test_seed'])
    assert checkpoint_key in CFG['checkpoints'] and variant in CFG['variants'] and seed in CFG['training_seeds'], (
        'Selecciona explícitamente final_test_checkpoint, final_test_variant y final_test_seed.')
    result = REPORT.get('experiments', {}).get(checkpoint_key, {}).get(variant, {}).get(str(seed))
    if result is None:
        raise RuntimeError('No están cargados los resultados de GPT-2 fine_tuned, semilla 123. Recupera la corrida del paso 09.')
    if not Path(result['checkpoint_dir']).is_dir():
        raise FileNotFoundError(f"No se encuentra el checkpoint entrenado: {result['checkpoint_dir']}")
    test_documents, test_audit = prepare_common_documents(official_test_ids, df_test)
    test_examples, test_stats = make_examples(checkpoint_key, test_documents, 'test')
    model = load_saved_model(result['checkpoint_dir'])
    test_args = TrainingArguments(output_dir=str(RUN / 'final_test_tmp'),
                                  per_device_eval_batch_size=CFG['batch_size'], report_to=[])
    test_trainer = Trainer(model=model, args=test_args, data_collator=collators[checkpoint_key],
                           processing_class=tokenizers[checkpoint_key],
                           compute_metrics=make_document_metrics(test_examples, test_documents),
                           preprocess_logits_for_metrics=preprocess_logits_for_metrics)
    try:
        test_metrics = test_trainer.evaluate(Dataset.from_list(model_features(test_examples)))
        final_test = {'checkpoint': checkpoint_key, 'variant': variant, 'seed': seed,
                      'metrics': extract_metrics(test_metrics), 'alignment': test_stats, 'gold_audit': test_audit,
                      'metric_scope': REPORT['metric_scope']}
        (RUN / f'final_test_{checkpoint_key}_{variant}_seed_{seed}.json').write_text(
            json.dumps(final_test, indent=2), encoding='utf-8')
        REPORT['final_test'] = final_test
        (RUN / 'summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
        mostrar_tabla_configuracion('Test final — GPT-2 fine_tuned, semilla 123', final_test)
    finally:
        del test_trainer, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


Ventanas gpt2 test:   0%|          | 0/250 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/150 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Step,Precision,Recall,F1,Support,Predicted,Accuracy,Invalid Bio Transitions,Precision Chemical,Recall Chemical,F1 Chemical,Support Chemical,Predicted Chemical,Precision Disease,Recall Disease,F1 Disease,Support Disease,Predicted Disease,Precision Procedure,Recall Procedure,F1 Procedure,Support Procedure,Predicted Procedure,Precision Protein,Recall Protein,F1 Protein,Support Protein,Predicted Protein,Precision Symptom,Recall Symptom,F1 Symptom,Support Symptom,Predicted Symptom
No log,1.098642,0,0.266712,0.371216,0.310405,9415,13104,0.810149,5612,0.560940,0.422099,0.481715,905,681,0.288802,0.434360,0.346932,2369,3563,0.322384,0.465605,0.380979,2951,4262,0.526786,0.126882,0.204506,465,112,0.145118,0.238899,0.180557,2725,4486


#### Test final — GPT-2 fine_tuned, semilla 123

| Parámetro | Valor |
| --- | --- |
| checkpoint | gpt2 |
| variant | fine_tuned |
| seed | 123 |
| metrics.micro.precision | 0.2667124542124542 |
| metrics.micro.recall | 0.3712161444503452 |
| metrics.micro.f1 | 0.3104045472711932 |
| metrics.micro.support | 9415 |
| metrics.micro.predicted | 13104 |
| metrics.per_label.CHEMICAL.precision | 0.5609397944199707 |
| metrics.per_label.CHEMICAL.recall | 0.4220994475138122 |
| metrics.per_label.CHEMICAL.f1 | 0.4817150063051703 |
| metrics.per_label.CHEMICAL.support | 905 |
| metrics.per_label.DISEASE.precision | 0.2888015717092338 |
| metrics.per_label.DISEASE.recall | 0.4343604896580836 |
| metrics.per_label.DISEASE.f1 | 0.346931894807822 |
| metrics.per_label.DISEASE.support | 2369 |
| metrics.per_label.PROCEDURE.precision | 0.32238385734396996 |
| metrics.per_label.PROCEDURE.recall | 0.465604879701796 |
| metrics.per_label.PROCEDURE.f1 | 0.38097878829890475 |
| metrics.per_label.PROCEDURE.support | 2951 |
| metrics.per_label.PROTEIN.precision | 0.5267857142857143 |
| metrics.per_label.PROTEIN.recall | 0.12688172043010754 |
| metrics.per_label.PROTEIN.f1 | 0.20450606585788564 |
| metrics.per_label.PROTEIN.support | 465 |
| metrics.per_label.SYMPTOM.precision | 0.14511814534106107 |
| metrics.per_label.SYMPTOM.recall | 0.23889908256880735 |
| metrics.per_label.SYMPTOM.f1 | 0.1805574816252947 |
| metrics.per_label.SYMPTOM.support | 2725 |
| metrics.loss | 1.0986422300338745 |
| metrics.accuracy | 0.8101488988736878 |
| metrics.invalid_bio_transitions | 5612 |
| alignment.documents | 250 |
| alignment.sequences | 385 |
| alignment.supervised_words | 99262 |
| alignment.retained | 9415 |
| alignment.gold_sha256 | 4e20072dd997ee8a52a921850f36f0a368bf8cdea86bb283fceb93ebeed2787f |
| alignment.policy | shared-spacy-first-subtoken-overlap-owner-document-v1 |
| alignment.seconds_this_run | 0.8536399229999461 |
| gold_audit.documents | 250 |
| gold_audit.annotations | 11239 |
| gold_audit.duplicates | 0 |
| gold_audit.unaligned | 136 |
| gold_audit.overlap_excluded | 1688 |
| gold_audit.retained | 9415 |
| gold_audit.words | 99791 |
| gold_audit.ignored_words | 529 |
| metric_scope | F1 micro de entidades exactas por documento y offsets de caracteres, gold común. |

### PASO 12 — Identificación en texto nuevo

Se utiliza el mismo tokenizador de palabras, ventanas, supervisión del primer subtoken y reconstrucción por documento que en entrenamiento. La semilla inicial es 123. Las entidades se presentan con texto, categoría y offsets; sus predicciones no sustituyen una evaluación clínica.


In [14]:
# PASO 12 - Inferencia con la misma preparación por palabras
TEXT_TO_IDENTIFY = ('Paciente con cefalea intensa y fiebre. Se indicó tratamiento '
                    'con paracetamol y se solicitó una resonancia magnética.')
IDENTIFICATION_CHECKPOINT = 'gpt2'
IDENTIFICATION_VARIANT = 'fine_tuned'
IDENTIFICATION_SEED = 123
assert isinstance(TEXT_TO_IDENTIFY, str) and TEXT_TO_IDENTIFY.strip()
result = REPORT.get('experiments', {}).get(IDENTIFICATION_CHECKPOINT, {}).get(
    IDENTIFICATION_VARIANT, {}).get(str(IDENTIFICATION_SEED))
assert result is not None, 'Ejecuta el paso 09 para la combinación elegida.'
tokens = [t for t in word_nlp(TEXT_TO_IDENTIFY) if not t.is_space]
assert tokens, 'El texto no contiene palabras.'
document = {'doc_id': 'nuevo', 'words': [t.text for t in tokens],
            'offsets': [(t.idx, t.idx + len(t)) for t in tokens],
            'labels': [0] * len(tokens), 'entities': []}
examples = build_word_windows(document, tokenizers[IDENTIFICATION_CHECKPOINT])
model = load_saved_model(result['checkpoint_dir']).to(device)
model.eval()
predictions = []
try:
    with torch.inference_mode():
        for start in range(0, len(examples), CFG['batch_size']):
            batch = collators[IDENTIFICATION_CHECKPOINT](model_features(examples[start:start+CFG['batch_size']]))
            batch.pop('labels')
            batch = {k: v.to(device) for k, v in batch.items()}
            predictions.extend(model(**batch).logits.argmax(dim=-1).cpu().tolist())
    words = reconstruct_word_predictions(predictions, examples, {'nuevo': document})['nuevo']
    spans, invalid = character_entities(words, document['offsets'])
    mostrar_tabla('Entidades detectadas', ['Texto', 'Categoría', 'Inicio', 'Fin'],
                  [[TEXT_TO_IDENTIFY[a:b], label, a, b] for a, b, label in sorted(spans)])
    display(Markdown(f'**Entidades:** {len(spans)}. **Transiciones BIO reparadas:** {invalid}.'))
finally:
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Loading weights:   0%|          | 0/150 [00:00<?, ?it/s]

#### Entidades detectadas

| Texto | Categoría | Inicio | Fin |
| --- | --- | --- | --- |
| cefalea | DISEASE | 13 | 20 |
| intensa | SYMPTOM | 21 | 28 |
| fiebre | SYMPTOM | 31 | 37 |
| tratamiento | PROCEDURE | 49 | 60 |
| paracetamol | CHEMICAL | 65 | 76 |
| resonancia magnética | PROCEDURE | 95 | 115 |

**Entidades:** 6. **Transiciones BIO reparadas:** 1.

# Informe final — Taller 4: reconocimiento de entidades clínicas con GPT-2 (clasificación de tokens)


## 1. Resultados principales del taller 4

| Modelo | Estrategia | Precisión | Recall | F1 | Mejor época | Épocas ejecutadas | Pasos | Tiempo de entrenamiento |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| GPT-2 (mrm8488/spanish-gpt2) | frozen | 22,598 % | 33,032 % | 26,837 % | 7 | 8 | 456 | 210,576 s |
| GPT-2 (mrm8488/spanish-gpt2) | fine_tuned | **26,948 %** | **37,349 %** | **31,307 %** | 8 | 8 | 456 | 494,009 s |

**GPT-2 fine_tuned obtuvo el mayor F1 observado: 31,307 %.** Su mejor checkpoint corresponde a la época 8, la última ejecutada: la corrida no alcanzó una meseta clara dentro del presupuesto de 8 épocas. GPT-2 frozen, en cambio, sí seleccionó una época anterior a la última (época 7 de 8), indicando una ligera caída de F1 en la época final.

Ambas variantes muestran recall sistemáticamente superior a precisión, igual que BETO y RoBERTa en el taller 3. El modelo predice más entidades de las que existen realmente (frozen: 121 entidades supervisadas predichas en la prueba técnica de 104 reales; ver patrón similar en validación), lo que es consistente con errores de delimitación de fronteras más que con un fallo sistemático de detección.

## 2. Efecto de la estrategia de adaptación

| Comparación con semilla 123 | Diferencia de F1 |
| --- | ---: |
| GPT-2: fine_tuned − frozen | **+4,471 pp** |

El ajuste completo mejora el F1 global, pero la magnitud de la mejora (+4,471 pp) es mucho menor que la observada en taller 3 para BETO (+25,444 pp) y RoBERTa (+28,356 pp). Esto es coherente con una arquitectura causal que, incluso con todos sus pesos actualizables, sigue sin poder atender al contexto a la derecha de cada palabra — una limitación estructural que el ajuste fino no puede compensar completamente.

El ajuste completo requirió aproximadamente **2,35 veces** el tiempo de frozen (494,009 s frente a 210,576 s), en línea con la proporción observada en taller 3 (~2,3–2,4×) para BETO y RoBERTa.

## 3. Desempeño por categoría clínica

| Categoría | Entidades de validación | GPT-2 frozen | GPT-2 fine_tuned |
| --- | ---: | ---: | ---: |
| CHEMICAL | 577 | **59,474 %** | 51,022 % |
| DISEASE | 1.391 | 26,751 % | **34,071 %** |
| PROCEDURE | 1.906 | 29,479 % | **38,180 %** |
| PROTEIN | 305 | **35,055 %** | 17,021 % |
| SYMPTOM | 1.682 | 14,782 % | **18,948 %** |
| **Total de entidades** | **5.861** | — | — |

**Hallazgo relevante:** a diferencia de taller 3 — donde el ajuste completo mejoró *todas* las categorías en ambos checkpoints —, en GPT-2 el ajuste completo **empeora** el F1 en CHEMICAL (−8,452 pp) y, de forma notable, en PROTEIN (−18,034 pp). El frozen conserva mejor estas dos categorías de menor soporte (577 y 305 entidades), mientras que el fine_tuned mejora las tres categorías de mayor soporte (DISEASE, PROCEDURE, SYMPTOM, que suman 4.979 entidades). Como el F1 global es **micro** (conteos agregados, no promedio simple de los cinco F1), la ganancia en las categorías mayoritarias domina la cifra global y hace que fine_tuned gane en conjunto pese a perder en dos categorías.

Esto sugiere que, para GPT-2 en esta configuración, el ajuste completo con `learning_rate=2e-5` puede estar sobreescribiendo representaciones que el encoder frozen ya aprovechaba razonablemente bien para entidades léxicamente más distintivas y de menor soporte (fármacos y proteínas), mientras gana capacidad general en categorías más frecuentes y heterogéneas (enfermedades, procedimientos, síntomas). No se investigó más a fondo si una tasa de aprendizaje menor evitaría esta regresión; queda como línea futura.

SYMPTOM sigue siendo, en ambas variantes, la categoría de menor F1 — igual que en taller 3 antes del ajuste completo, aunque en GPT-2 el valor absoluto (14,782 % / 18,948 %) es muy inferior al de BETO/RoBERTa fine_tuned (~56–58 %).

## 4. Comparación de los cuatro talleres

| Taller | Modelo y variante | Semillas | Precisión | Recall | F1 de validación |
| --- | --- | --- | ---: | ---: | ---: |
| 1 | Bi-LSTM con POS | 42, 123, 2026 | 32,44 % | 48,76 % | **38,96 % ± 0,51** |
| 1 | Bi-LSTM sin POS | 42, 123, 2026 | 30,90 % | 46,53 % | 37,13 % ± 1,30 |
| 2 | Transformer desde cero con POS | 42, 123, 2026 | 19,94 % | 40,54 % | **26,69 % ± 1,73** |
| 2 | Transformer desde cero sin POS | 42, 123, 2026 | 18,21 % | 40,14 % | 24,98 % ± 2,26 |
| 3 | BETO frozen | 123 | 31,922 % | 47,739 % | 38,261 % |
| 3 | BETO fine_tuned | 123 | 59,716 % | 68,265 % | **63,705 %** |
| 3 | RoBERTa clínico frozen | 123 | 29,284 % | 42,450 % | 34,659 % |
| 3 | RoBERTa clínico fine_tuned | 123 | 58,194 % | 68,708 % | **63,015 %** |
| 4 | GPT-2 frozen | 123 | 22,598 % | 33,032 % | 26,837 % |
| 4 | GPT-2 fine_tuned | 123 | 26,948 % | 37,349 % | 31,307 % |

GPT-2 fine_tuned (31,307 %) **supera** a ambos Transformers desde cero del taller 2 (24,98 % sin POS, 26,69 % con POS, medias multisemilla) por +6,327 pp y +4,617 pp respectivamente, pero queda **7,653 pp por debajo** de la Bi-LSTM con POS del taller 1 (38,96 %) y **32,398 pp por debajo** de BETO fine_tuned del taller 3 (63,705 %). GPT-2 frozen (26,837 %) también queda por encima de ambos Transformers desde cero del taller 2 y prácticamente iguala al mejor de ellos (con POS, 26,69 %), pese a partir de pesos preentrenados generales en vez de entrenar desde cero — mientras que BETO y RoBERTa frozen (34,659–38,261 %) superan holgadamente a GPT-2 frozen.

Esta comparación es descriptiva, no controlada: taller 4 usa una sola semilla (123, igual que taller 3), mientras talleres 1 y 2 promedian tres semillas. Además, taller 4 comparte con taller 3 el protocolo de evaluación por documento con ventanas solapadas (`entity_spans`/`scores_from_counts`, reconstrucción por palabra propietaria), distinto del protocolo de talleres 1 y 2.

El resultado más informativo de esta comparación es **arquitectónico, no solo numérico**: entre los dos modelos preentrenados de tamaño comparable evaluados con el mismo protocolo (BETO/RoBERTa en taller 3 y GPT-2 en taller 4), los encoders bidireccionales superan ampliamente al decoder causal en esta tarea de NER, tanto en frozen como en fine_tuned.

## 5. Modelo seleccionado y evaluación final en test

Se seleccionó **GPT-2 fine_tuned, semilla 123**, por obtener el mayor F1 de validación de los dos escenarios: **31,307 %**, utilizando el checkpoint de la **época 8** (la última ejecutada). El paso 11 evaluó este modelo guardado sobre los **250 documentos de test**, sin volver a entrenarlo.

### Resultado global y comparación con validación

| Métrica | Validación | Test | Cambio test − validación |
| --- | ---: | ---: | ---: |
| Precisión micro | 26,948 % | 26,671 % | −0,277 pp |
| Recall micro | 37,349 % | 37,122 % | −0,227 pp |
| F1 micro | 31,307 % | 31,040 % | −0,267 pp |

El modelo obtuvo un **F1 de test de 31,040 %**, prácticamente idéntico al de validación (diferencia de apenas −0,267 pp). Esta estabilidad es una señal positiva: el modelo no sobreajustó a la partición de validación durante la selección de época.

Se evaluaron **9.415 entidades de referencia** y se predijeron **13.104 entidades** — una sobrepredicción más marcada que la de BETO en taller 3 (10.782 sobre 9.415). La exactitud por palabra fue **81,015 %**, inferior al 88,049 % de BETO fine_tuned, y la pérdida de test fue **1,098642**, notablemente mayor que la de BETO (0,748552).

### Resultado de test por categoría

| Categoría | Entidades de referencia | Precisión | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| CHEMICAL | 905 | 56,094 % | 42,210 % | **48,172 %** |
| DISEASE | 2.369 | 28,880 % | 43,436 % | 34,693 % |
| PROCEDURE | 2.951 | 32,238 % | 46,560 % | 38,098 % |
| PROTEIN | 465 | 52,679 % | 12,688 % | 20,451 % |
| SYMPTOM | 2.725 | 14,512 % | 23,890 % | 18,056 % |

El patrón de validación se repite en test: **CHEMICAL** mantiene el mayor F1 (**48,172 %**) y **SYMPTOM** el menor (**18,056 %**). **PROTEIN** muestra el desequilibrio más marcado entre precisión (52,679 %) y recall (12,688 %) de las cinco categorías — el modelo es selectivo pero pierde la mayoría de las proteínas reales, consistente con lo observado en validación (donde fine_tuned también perdía terreno en PROTEIN frente a frozen).

### Alcance de la evaluación

Los 250 documentos se procesaron en **385 ventanas**, con **99.262 palabras supervisadas**. De las **11.239 anotaciones originales** del test, se conservaron **9.415**: se excluyeron **136** por falta de alineación y **1.688** por solapamiento — cifras idénticas a las de taller 3, como corresponde a compartir el mismo gold común independiente del tokenizador.

Se registraron **5.612 transiciones BIO inválidas**, casi 5 veces las de BETO en taller 3 (1.205), tratadas por la métrica como inicios de entidad. Este volumen de transiciones inválidas es consistente con una atención causal que dificulta más la delimitación coherente de los límites de entidad.

## 6. Consideraciones específicas de GPT-2 como clasificador de tokens

A diferencia de BETO y RoBERTa (taller 3), que son encoders **bidireccionales** (cada posición atiende tanto al contexto izquierdo como al derecho), GPT-2 es un decoder autoregresivo con atención estrictamente **causal**: cada posición solo puede atender a las palabras que la preceden. Los resultados de esta corrida son consistentes con esa limitación estructural: F1 de validación de GPT-2 fine_tuned (31,307 %) muy por debajo de BETO/RoBERTa fine_tuned (~63 %), y un número de transiciones BIO inválidas en test casi 5 veces mayor (5.612 frente a 1.205) — ambos indicadores de mayor dificultad para delimitar fronteras de entidad sin contexto derecho.

Un hallazgo adicional no anticipado: el ajuste completo **no** mejora todas las categorías en GPT-2, a diferencia de BETO/RoBERTa donde el fine-tuning fue monótonamente positivo. CHEMICAL y PROTEIN — las dos categorías de menor soporte — obtuvieron mejor F1 con el encoder **congelado**. Esto podría indicar que, con atención causal, actualizar todos los pesos del modelo compromete representaciones léxicas específicas en favor de patrones más generales y frecuentes, un efecto que no se observa (o se observa en mucha menor medida) con atención bidireccional.

Este taller usa un único checkpoint (`mrm8488/spanish-gpt2`) porque no existe actualmente una contraparte de GPT-2 en español preentrenada específicamente en dominio clínico o biomédico. Por tanto, no es posible replicar aquí la comparación de dominio general-vs-clínico que sí ofrece el taller 3 (BETO vs. RoBERTa clínico).

Cabe señalar además una incidencia técnica real encontrada al adaptar el checkpoint: el tokenizer publicado de `mrm8488/spanish-gpt2` declara un vocabulario (50.265 ids, incluyendo `<|endoftext|>` como token añadido en el id 50265) más grande que la matriz de embeddings real del modelo (`config.vocab_size=50257`), lo que provocaba un error de acceso fuera de rango en GPU al tokenizar cualquier texto. Se resolvió redimensionando la matriz de embeddings (`resize_token_embeddings`) tras cargar el modelo, una técnica estándar para esta situación. Las filas añadidas se inicializan de forma aleatoria (sin pesos preentrenados), lo que incluye la representación del propio token de frontera `<|endoftext|>` usado para delimitar cada ventana — una limitación adicional específica de este checkpoint que no aplica a BETO/RoBERTa en taller 3.

## 7. Limitaciones y conclusión

La comparación entre los cuatro talleres comparte corpus y partición, pero no un protocolo de evaluación único: talleres 1 y 2 promedian tres semillas con un protocolo propio, mientras talleres 3 y 4 usan una sola semilla con el protocolo de ventanas solapadas y reconstrucción por documento. La comparación histórica es orientativa. Los tiempos de entrenamiento tampoco deben extrapolarse como benchmark de eficiencia entre arquitecturas.

**El resultado principal del taller 4 es que GPT-2, usado como clasificador de tokens, queda muy por debajo de los encoders bidireccionales del taller 3 en esta tarea de NER clínico**, incluso después de ajuste completo: F1 de validación de 31,307 % frente a 63,015–63,705 % de BETO/RoBERTa fine_tuned. La diferencia es consistente con la limitación estructural de la atención causal (sin contexto derecho) más que con un problema de optimización o de tamaño de modelo — GPT-2 pequeño (124 M de parámetros) es comparable en escala a BETO (110 M) y RoBERTa clínico, por lo que la brecha de rendimiento no se explica por capacidad del modelo. Un segundo hallazgo relevante es que, a diferencia de taller 3, el ajuste completo de GPT-2 no mejora uniformemente todas las categorías: empeora CHEMICAL y PROTEIN frente a la variante frozen, incluso mientras mejora el F1 global. Frente a los talleres anteriores, estos resultados respaldan que, para NER por clasificación de tokens, los encoders bidireccionales siguen siendo la opción preferible frente a un decoder causal de tamaño comparable.
